# E12 · дообучение адаптера через выход замороженной B5

**Один ограниченный эксперимент:** стартуем с твоих обученных весов E9-k4 (227 008 параметров), архитектуру не меняем. B0 и B5 заморожены, B5 в eval. Учитель — обычная B5; ученик — та же B5 с предсказанными контекстами и final. Обновляется только адаптер.

Сначала CPU-гейт по готовому кэшу, затем (если пройден) дообучение максимум 5 эпох. Старые веса/результаты не перезаписываются. Новый каталог: `Мой диск/e8_cityscapes_v1/e12_output_distill_v1/`.

Открой notebook в Colab, выбери GPU, сохрани копию на Drive, выполняй сверху вниз. Путь к архивам уже `datasets/cityscapes`. Подготовка восстановит E8, но не повторяет E0/E8/E9. Нужны прежние папки `e9_k4_adapter_v1` (включая targets), `e8c_k_sweep_v1` и Cityscapes val.

Это по-прежнему teacher-assisted схема с настоящими ранними стадиями B5 и входом выбранных токенов. **Не доказательство ускорения. ACDC не используется.** Повторное использование cs_test для гейта/оценки делает эксперимент исследовательским.


In [ ]:
# 1. Окружение. PyTorch/CUDA не переустанавливаем.
import os, sys, subprocess
os.environ['USE_TF'] = '0'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.46.3', 'cityscapesscripts==2.2.4', 'scikit-learn', 'tqdm'], check=True)
import torch
assert torch.cuda.is_available(), 'Включи GPU: Среда выполнения → Сменить среду выполнения'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2. Укажи папку с архивами. Обычно нужно поменять ТОЛЬКО эту строку.
from pathlib import Path
ARCHIVE_DIR = Path('/content/drive/MyDrive/datasets/cityscapes')  # <-- твоя папка или ярлык
IMAGE_ZIP = ARCHIVE_DIR / 'val.zip'
GT_ZIP = ARCHIVE_DIR / 'gtFine_trainvaltest.zip'

# Если архив называется иначе, поменяй IMAGE_ZIP/GT_ZIP выше.
for p in [IMAGE_ZIP, GT_ZIP]:
    assert p.is_file(), f'Не найден {p}. Найди архив в панели Files → drive → MyDrive и скопируй путь.'

RUN_NAME = 'e8_cityscapes_v1'  # для нового независимого запуска используй другое имя
SAVE = Path('/content/drive/MyDrive') / RUN_NAME
WORK = Path('/content') / RUN_NAME
EXP = WORK / 'experiments'
DATA = WORK / 'data'
CACHE = WORK / 'cache'
RESULTS = SAVE / 'results'
for p in [SAVE, EXP, DATA, CACHE, RESULTS]: p.mkdir(parents=True, exist_ok=True)
os.environ.update(E8_DATA=str(DATA), E8_CACHE=str(CACHE), E8_RESULTS=str(RESULTS), E8_BACKUP=str(SAVE/'backups'))
print('Результаты:', SAVE)


## Распаковка только validation
Поддерживаются `val/<город>/...` и `leftImg8bit/val/<город>/...` в архиве изображений. Из разметки берём только `gtFine/val/*/*_labelIds.png`. Полный train/test не распаковывается.

In [ ]:
# 3. Извлечение и проверка 500 пар.
import zipfile, shutil
from collections import Counter

def unpack_cityscapes(archive, image):
    count = 0
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            parts = Path(member.filename).parts
            suffix = '_leftImg8bit.png' if image else '_gtFine_labelIds.png'
            if member.is_dir() or not member.filename.endswith(suffix) or 'val' not in parts:
                continue
            k = parts.index('val')
            tail = parts[k:]
            if len(tail) != 3 or '..' in tail: continue
            branch = 'leftImg8bit' if image else 'gtFine'
            out = DATA/'cityscapes'/branch/Path(*tail)
            out.parent.mkdir(parents=True, exist_ok=True)
            if not out.exists() or out.stat().st_size != member.file_size:
                tmp = out.with_suffix(out.suffix+'.part')
                with z.open(member) as src, tmp.open('wb') as dst: shutil.copyfileobj(src,dst)
                tmp.replace(out)
            count += 1
    return count

print('Изображений:', unpack_cityscapes(IMAGE_ZIP, True))
print('Масок:', unpack_cityscapes(GT_ZIP, False))
images = sorted((DATA/'cityscapes/leftImg8bit/val').glob('*/*_leftImg8bit.png'))
counts = Counter(p.parent.name for p in images)
assert dict(counts) == {'frankfurt':267, 'lindau':59, 'munster':174}, counts
for p in images:
    gt = DATA/'cityscapes/gtFine/val'/p.parent.name/p.name.replace('_leftImg8bit.png','_gtFine_labelIds.png')
    assert gt.is_file(), f'Отсутствует маска {gt}'
print('500 пар OK:', dict(counts))


In [ ]:
# 4. Создание автономных скриптов (никакие файлы репозитория не меняются).
import json, hashlib
SOURCES = {'common.py': 'import os\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\nfrom transformers import SegformerForSemanticSegmentation\nfrom cityscapesscripts.helpers.labels import labels\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ["E8_DATA"])\nCACHE = Path(os.environ["E8_CACHE"])\nRESULTS = Path(os.environ["E8_RESULTS"])\n\nSIZE = (512, 1024)\nNUM_CLASSES = 19\nIGNORE = 255\n\nMODELS = {\n    "b0": "nvidia/segformer-b0-finetuned-cityscapes-512-1024",\n    "b5": "nvidia/segformer-b5-finetuned-cityscapes-1024-1024",\n}\n\n# frankfurt is used to fit scores/routers, the rest is test only\nDOMAINS = ["cs_fit", "cs_test", "fog", "night", "rain", "snow"]\n\n_LUT = np.full(256, IGNORE, np.uint8)\nfor l in labels:\n    if 0 <= l.id < 256 and l.trainId not in (-1, 255):\n        _LUT[l.id] = l.trainId\n\n\ndef list_domain(domain):\n    """Returns sorted (image, gt) path pairs for a domain."""\n    if domain.startswith("cs_"):\n        cities = ["frankfurt"] if domain == "cs_fit" else ["lindau", "munster"]\n        imgs = [p for c in cities for p in (DATA / "cityscapes/leftImg8bit/val" / c).glob("*.png")]\n        gt = lambda p: Path(str(p).replace("leftImg8bit", "gtFine", 1)\n                            .replace("_leftImg8bit.png", "_gtFine_labelIds.png"))\n    else:\n        imgs = list((DATA / "acdc/rgb_anon" / domain / "val").rglob("*_rgb_anon.png"))\n        gt = lambda p: Path(str(p).replace("rgb_anon", "gt", 1)\n                            .replace("_rgb_anon.png", "_gt_labelTrainIds.png"))\n    return [(p, gt(p)) for p in sorted(imgs)]\n\n\ndef image_tensor(image):\n    # Tensor bilinear antialiased resize, matching the original torchvision v2 intent.\n    import torch.nn.functional as F\n    a = np.array(image, dtype=np.uint8, copy=True)\n    x = torch.from_numpy(a).permute(2,0,1).float().div(255)\n    x = F.interpolate(x[None], size=SIZE, mode="bilinear", align_corners=False, antialias=True)[0]\n    mean = torch.tensor([.485,.456,.406])[:,None,None]\n    std = torch.tensor([.229,.224,.225])[:,None,None]\n    return (x-mean)/std\n\n\nclass SegSet(Dataset):\n    def __init__(self, domain):\n        self.domain = domain\n        self.items = list_domain(domain)\n        self.img_tf = image_tensor\n\n\n    def __len__(self):\n        return len(self.items)\n\n    def __getitem__(self, i):\n        img_path, gt_path = self.items[i]\n        gt = np.array(Image.open(gt_path))\n        if self.domain.startswith("cs_"):\n            gt = _LUT[gt]\n        gt = Image.fromarray(gt).resize(SIZE[::-1], Image.NEAREST)\n        return self.img_tf(Image.open(img_path).convert("RGB")), torch.from_numpy(np.array(gt)), img_path.stem\n\n\ndef load_model(name, device="cuda"):\n    model = SegformerForSemanticSegmentation.from_pretrained(MODELS[name])\n    return model.half().to(device).eval()\n\n\ndef confusion(pred, gt):\n    m = gt != IGNORE\n    return np.bincount(NUM_CLASSES * gt[m].astype(np.int64) + pred[m],\n                       minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)\n\n\ndef miou(conf):\n    iou = np.diag(conf) / (conf.sum(0) + conf.sum(1) - np.diag(conf) + 1e-9)\n    return 100 * iou[conf.sum(1) > 0].mean()\n\n\nTILE = 64  # 512x1024 -> 8x16 tiles\n\n\ndef load(domain, *keys):\n    """Yields cached arrays of every image in a domain, in a fixed order."""\n    for p in sorted((CACHE / domain).glob("*.npz")):\n        with np.load(p) as z:\n            yield [z[k] for k in keys]\n\n\ndef tile_sum(x):\n    h, w = x.shape\n    return x.reshape(h // TILE, TILE, w // TILE, TILE).sum((1, 3))\n\n\ndef gain(gt, pred_b0, pred_b5):\n    """+1 where only B5 is right, -1 where only B0 is right, 0 elsewhere or on ignore."""\n    return ((pred_b5 == gt).astype(np.int8) - (pred_b0 == gt)) * (gt != IGNORE)\n\n\ndef top_tiles(scores, budget):\n    """Boolean mask of the top `budget` share of tiles by score, over the whole domain."""\n    order = np.argsort(-scores.ravel(), kind="stable")\n    m = np.zeros(scores.size, bool)\n    m[order[:round(budget * scores.size)]] = True\n    return m.reshape(scores.shape)\n\n\ndef route_confs(domain, scores, budgets):\n    """Per-image confusions (budgets x images x C x C) when top_tiles get B5."""\n    masks = [top_tiles(scores, b) for b in budgets]\n    confs = np.zeros((len(budgets), len(scores), NUM_CLASSES, NUM_CLASSES), np.int64)\n    for i, (gt, b0, b5) in enumerate(load(domain, "gt", "pred_b0", "pred_b5")):\n        for j, m in enumerate(masks):\n            px = m[i].repeat(TILE, 0).repeat(TILE, 1)\n            confs[j, i] = confusion(np.where(px, b5, b0), gt)\n    return confs\n\n\ndef route_curve(domain, scores, budgets):\n    """mIoU per budget, see route_confs."""\n    return [miou(c.sum(0)) for c in route_confs(domain, scores, budgets)]\n', 'e2_confidence.py': '"""E2: standard confidence scores of B0 as routing signals.\n\nPixel AUROC (logit resolution, sampled pixels) for two targets:\n  err  - B0 is wrong\n  help - B0 is wrong and B5 is right\nTile routing: mean score per tile, % of the B5-B0 gap closed (as in E1).\n"""\nimport json\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom sklearn.metrics import roc_auc_score\n\nfrom common import DOMAINS, IGNORE, RESULTS, TILE, load, route_curve\n\nBUDGETS = [0.05, 0.1, 0.2, 0.3]\nPIXELS_PER_IMAGE = 2000\nSTRIDE = 4  # logits are at 1/4 resolution\n\n\ndef uncertainty(logits):\n    """Per-pixel scores, higher = less confident."""\n    p = logits.softmax(0)\n    top2 = p.topk(2, dim=0).values\n    return {\n        "entropy": -(p * p.clamp_min(1e-12).log()).sum(0),\n        "msp": 1 - top2[0],\n        "margin": top2[1] - top2[0],\n        "energy": -logits.logsumexp(0),\n    }\n\n\ndef main():\n    base = json.loads((RESULTS / "e0.json").read_text())\n    oracle = json.loads((RESULTS / "e1.json").read_text())\n    rng = np.random.default_rng(0)\n    auroc, closed = {}, {}\n\n    for domain in DOMAINS:\n        tiles, pix, err, helps = {}, {}, [], []\n        for gt, b0, b5, logits in load(domain, "gt", "pred_b0", "pred_b5", "logits_b0"):\n            scores = uncertainty(torch.from_numpy(logits).float())\n            for k, s in scores.items():\n                tiles.setdefault(k, []).append(F.avg_pool2d(s[None], TILE // STRIDE)[0].numpy())\n\n            gt, b0, b5 = (a[STRIDE // 2::STRIDE, STRIDE // 2::STRIDE] for a in (gt, b0, b5))\n            valid = np.flatnonzero(gt != IGNORE)\n            idx = rng.choice(valid, min(PIXELS_PER_IMAGE, valid.size), replace=False)\n            for k, s in scores.items():\n                pix.setdefault(k, []).append(s.numpy().ravel()[idx])\n            wrong = b0.ravel()[idx] != gt.ravel()[idx]\n            err.append(wrong)\n            helps.append(wrong & (b5.ravel()[idx] == gt.ravel()[idx]))\n\n        err, helps = np.concatenate(err), np.concatenate(helps)\n        b0_miou, b5_miou = base[domain]["b0"], base[domain]["b5"]\n        auroc[domain], closed[domain] = {}, {}\n        for k in tiles:\n            s = np.concatenate(pix[k])\n            auroc[domain][k] = [round(roc_auc_score(err, s), 3), round(roc_auc_score(helps, s), 3)]\n            curve = route_curve(domain, np.stack(tiles[k]), BUDGETS)\n            closed[domain][k] = [round(100 * (m - b0_miou) / (b5_miou - b0_miou), 1) for m in curve]\n\n    print(f"{\'domain\':8} {\'method\':8} {\'AUC err\':>8} {\'AUC help\':>9}" + "".join(f"{b:>7.0%}" for b in BUDGETS))\n    for domain in DOMAINS:\n        for k, (a_err, a_help) in auroc[domain].items():\n            print(f"{domain:8} {k:8} {a_err:8.3f} {a_help:9.3f}" + "".join(f"{c:7.1f}" for c in closed[domain][k]))\n        print(f"{domain:8} {\'oracle\':8} {\'\':8} {\'\':9}" + "".join(f"{c:7.1f}" for c in oracle[domain]["oracle_tile"][:len(BUDGETS)]))\n\n    (RESULTS / "e2.json").write_text(json.dumps({"budgets": BUDGETS, "auroc": auroc, "closed": closed}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'e8_cityscapes.py': '"""E8: gate for the adapter idea - is a frozen stage-3 context good enough for B5?\n\nB5 sees the whole frame, but in the chosen stages only the tokens of routed tiles are updated\nby the transformer blocks. Every other token keeps its stage-entry value for all blocks, so it\nonly serves as keys/values ("frozen context"). Here the entry values come from the real B5\n(an upper bound for any adapter that maps B0 tokens into B5 token space).\n\nQuality only: the whole frame is still computed, only the write-back is masked. The saving is\ncounted analytically for now (blocks run on routed queries only; context costs K/V projections),\nreal FLOPs and latency are measured later.\n\nStages (0-based): 2 = stride 16, 40 blocks, most of B5\'s encoder compute; 3 = stride 32.\nVariants: "s3" freezes stage 2 only, "s3+s4" freezes stages 2 and 3. Stages 0-1 are run on the\nwhole frame in this experiment (known simplification, about a fifth of the encoder cost).\n\nTiles are selected as in E5 (domain-wide ranking at 10% and 20%): `rank` (entropy + knn) and the oracle.\nReported as % of the B5-B0 gap closed, next to:\n  full  - B5 on the full frame inside the routed tiles (the E5 "full" column, the target)\n  agree - pixel agreement with that full-frame B5 prediction inside the routed tiles\nGate: s3 should stay close to `full` on ACDC. For reference E5 ctx64 (cost x0.9) closed 42 / 17 / 35 / 60\n(fog / night / rain / snow) against 61 / 30 / 59 / 59 for full.\n"""\nimport json\nfrom types import SimpleNamespace\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\n\nfrom common import (CACHE, DOMAINS, NUM_CLASSES, RESULTS, SIZE, TILE, SegSet, confusion, gain, load,\n                    load_model, miou, tile_sum, top_tiles)\ndef rank(scores):\n    return scores.ravel().argsort().argsort().reshape(scores.shape)\n\nDOMAINS = ["cs_fit", "cs_test"]\n\nBUDGETS = [0.1, 0.2]\nSTAGES = {"s3": [2], "s3+s4": [2, 3]}\n\n\nclass Frozen(nn.Module):\n    """Wraps a SegformerLayer: tokens outside state.mask are reset to the stage entry value."""\n\n    def __init__(self, layer, state, first):\n        super().__init__()\n        self.layer, self.state, self.first = layer, state, first\n\n    def forward(self, hidden_states, *args, **kwargs):\n        s = self.state\n        if self.first:\n            s.entry = hidden_states\n        out = self.layer(hidden_states, *args, **kwargs)\n        if s.mask is None:\n            return out\n        return (torch.where(s.mask, out[0], s.entry),) + tuple(out[1:])\n\n\ndef install(model, stages=(2, 3)):\n    """Wrap every block of the given stages, returns {stage: state}."""\n    state = {st: SimpleNamespace(mask=None, entry=None) for st in stages}\n    blocks = model.segformer.encoder.block\n    for st in stages:\n        for i, layer in enumerate(blocks[st]):\n            blocks[st][i] = Frozen(layer, state[st], i == 0)\n    return state\n\n\ndef token_mask(m, stage):\n    """(8, 16) bool tile mask -> (1, N, 1) bool mask over the tokens of a stage (stride 4 * 2**stage)."""\n    k = TILE // (4 * 2 ** stage)\n    t = torch.from_numpy(m).cuda().repeat_interleave(k, 0).repeat_interleave(k, 1)\n    return t.reshape(1, -1, 1)\n\n\n@torch.inference_mode()\ndef predict(model, state, img, masks):\n    """B5 prediction on the full frame where only `masks` {stage: token mask} tokens are updated."""\n    for st, s in state.items():\n        s.mask = masks.get(st)\n    try:\n        logits = model(pixel_values=img[None].cuda().half()).logits\n        return F.interpolate(logits.float(), SIZE, mode="bilinear", align_corners=False).argmax(1)[0].byte().cpu().numpy()\n    finally:\n        for s in state.values():\n            s.mask = None\n            s.entry = None\n\n\n\ndef sanity(model, state):\n    """An all-true mask must reproduce the cached full-frame B5 prediction."""\n    img = SegSet("cs_fit")[0][0]\n    (cached,) = next(load("cs_fit", "pred_b5"))\n    full = {st: token_mask(np.ones((SIZE[0] // TILE, SIZE[1] // TILE), bool), st) for st in state}\n    agree = (predict(model, state, img, full) == cached).mean()\n    assert agree > 0.999, f"all-true mask does not reproduce B5, agreement {agree:.4f}"\n\n\ndef main():\n    model = load_model("b5")\n    state = install(model)\n    sanity(model, state)\n\n    base = json.loads((RESULTS / "e0_cityscapes.json").read_text())\n    feats = np.load(CACHE / "e8_tiles_cityscapes.npz")\n    report_path = RESULTS / "e8_cityscapes.json"\n    report = json.loads(report_path.read_text()) if report_path.exists() else {}\n\n    print(f"{\'domain\':8} {\'method\':8} {\'budget\':>6} {\'full\':>7}"\n          + "".join(f"{n:>9}" for n in STAGES) + "".join(f"{f\'agree {n}\':>14}" for n in STAGES))\n    for domain in DOMAINS:\n        if domain in report:\n            print(domain, "уже завершён, пропускаем", flush=True)\n            continue\n        dataset = SegSet(domain)\n        assert [p.stem for p, _ in dataset.items] == [p.stem for p in sorted((CACHE / domain).glob("*.npz"))]\n\n        entropy, knn = feats[f"{domain}__entropy__mean"], feats[f"{domain}__knn__mean"]\n        gains = np.stack([tile_sum(gain(*a)) for a in load(domain, "gt", "pred_b0", "pred_b5")])\n        scores = {"rank": rank(entropy) + rank(knn), "oracle": gains}\n        masks = {(k, b): top_tiles(s, b) for k, s in scores.items() for b in BUDGETS}\n\n        zeros = lambda: np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)\n        full = {key: zeros() for key in masks}\n        conf = {(*key, n): zeros() for key in masks for n in STAGES}\n        agree = {(*key, n): np.zeros(2, np.int64) for key in masks for n in STAGES}\n\n        started = __import__("time").monotonic()\n        loader = DataLoader(dataset, batch_size=None, num_workers=2)\n        for i, ((img, _, _), (gt, b0, b5)) in enumerate(zip(loader, load(domain, "gt", "pred_b0", "pred_b5"))):\n            for key, m in masks.items():\n                px = m[i].repeat(TILE, 0).repeat(TILE, 1)\n                full[key] += confusion(np.where(px, b5, b0), gt)\n                for n, stages in STAGES.items():\n                    pred = predict(model, state, img, {st: token_mask(m[i], st) for st in stages})\n                    conf[(*key, n)] += confusion(np.where(px, pred, b0), gt)\n                    agree[(*key, n)] += [(pred == b5)[px].sum(), px.sum()]\n\n            if (i+1) % 5 == 0 or i+1 == len(dataset):\n                elapsed = __import__("time").monotonic()-started\n                eta = elapsed/(i+1)*(len(dataset)-i-1)/60\n                print(f"E8 {domain}: {i+1}/{len(dataset)}, {elapsed/60:.1f} мин, ETA ~{eta:.1f} мин", flush=True)\n\n        b0_miou, b5_miou = base[domain]["b0"], base[domain]["b5"]\n        closed = lambda c: round(100 * (miou(c) - b0_miou) / (b5_miou - b0_miou), 1)\n        report[domain] = {}\n        for key in masks:\n            row = {"full": closed(full[key])}\n            row.update({n: closed(conf[(*key, n)]) for n in STAGES})\n            row.update({f"agree_{n}": round(100 * agree[(*key, n)][0] / agree[(*key, n)][1], 1) for n in STAGES})\n            report[domain][f"{key[0]}@{key[1]}"] = row\n            print(f"{domain:8} {key[0]:8} {key[1]:6.0%} {row[\'full\']:7.1f}"\n                  + "".join(f"{row[n]:9.1f}" for n in STAGES)\n                  + "".join(f"{row[f\'agree_{n}\']:14.1f}" for n in STAGES))\n\n        temp = report_path.with_suffix(".json.part")\n        temp.write_text(json.dumps(report, indent=2))\n        temp.replace(report_path)\n        print("Сохранён домен:", domain, flush=True)\n\n    print("Результат:", report_path, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'runner.py': '"""Cityscapes-only preparation and E8; frozen B0/B5. No ACDC tuning."""\nimport os\nos.environ[\'USE_TF\'] = \'0\'\nimport argparse, gc, hashlib, json, platform, tarfile, time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom common import CACHE, RESULTS, SIZE, TILE, SegSet, load, load_model, confusion, miou\n\nDOMAINS = [\'cs_fit\', \'cs_test\']\nSIGNATURE = \'e8-cs-colab-v1\'\n\ndef write_json(path, data):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.part\')\n    tmp.write_text(json.dumps(data, indent=2))\n    tmp.replace(path)\n\ndef checkpoint(domain):\n    backup = os.environ.get(\'E8_BACKUP\')\n    if not backup:\n        return\n    dest = Path(backup); dest.mkdir(parents=True, exist_ok=True)\n    name = dest / f\'cache_{domain}.tar\'\n    tmp = name.with_suffix(\'.tar.part\')\n    with tarfile.open(tmp, \'w\') as tar:\n        tar.add(CACHE / domain, arcname=domain)\n    tmp.replace(name)\n    print(\'Кэш сохранён на Drive:\', name, flush=True)\n\ndef valid_cache(path):\n    if not path.exists():\n        return False\n    with np.load(path, allow_pickle=False) as z:\n        assert set([\'gt\', \'pred_b0\', \'pred_b5\', \'logits_b0\', \'feat_b0\']) <= set(z.files)\n        assert all(z[k].shape == SIZE for k in [\'gt\', \'pred_b0\', \'pred_b5\'])\n        assert z[\'logits_b0\'].shape == (19, 128, 256)\n        assert z[\'feat_b0\'].shape == (256, 64, 128)\n    return True\n\n@torch.inference_mode()\ndef cache():\n    models = [load_model(n) for n in [\'b0\', \'b5\']]\n    captured = {}\n    handle = models[0].decode_head.classifier.register_forward_pre_hook(lambda m, x: captured.update(f=x[0]))\n    report = {}\n    try:\n        for domain in DOMAINS:\n            data = SegSet(domain)\n            assert len(data) == (267 if domain == \'cs_fit\' else 233)\n            folder = CACHE / domain; folder.mkdir(parents=True, exist_ok=True)\n            confs = [np.zeros((19,19), np.int64) for _ in range(2)]\n            start = time.monotonic()\n            for i in range(len(data)):\n                img, gt, name = data[i]\n                path = folder / f\'{name}.npz\'\n                if not valid_cache(path):\n                    x = img[None].cuda().half()\n                    l0 = models[0](pixel_values=x).logits\n                    feat = F.avg_pool2d(captured[\'f\'], 2)[0].cpu().numpy()\n                    preds = []\n                    for logits in [l0, models[1](pixel_values=x).logits]:\n                        preds.append(F.interpolate(logits.float(), SIZE, mode=\'bilinear\', align_corners=False).argmax(1)[0].byte().cpu().numpy())\n                    tmp = path.with_suffix(\'.npz.part\')\n                    with tmp.open(\'wb\') as f:\n                        np.savez_compressed(f, gt=gt.numpy(), pred_b0=preds[0], pred_b5=preds[1], logits_b0=l0[0].cpu().numpy(), feat_b0=feat)\n                    tmp.replace(path)\n                with np.load(path) as z:\n                    for j, k in enumerate([\'pred_b0\', \'pred_b5\']):\n                        confs[j] += confusion(z[k], z[\'gt\'])\n                if (i+1) % 10 == 0 or i+1 == len(data):\n                    print(f\'E0 {domain}: {i+1}/{len(data)}, {(time.monotonic()-start)/60:.1f} мин\', flush=True)\n            report[domain] = dict(zip([\'b0\',\'b5\'], [float(miou(c)) for c in confs]))\n            write_json(RESULTS / \'e0_cityscapes.json\', report)\n            checkpoint(domain)\n    finally:\n        handle.remove()\n    print(report, flush=True)\n\n@torch.inference_mode()\ndef scores():\n    """Only entropy + kNN required by E8. Bank fitted solely on cs_fit."""\n    from e2_confidence import uncertainty\n    from common import IGNORE\n    reference = CACHE / \'e8_knn_bank.npy\'\n    if not reference.exists():\n        rng = np.random.default_rng(0)\n        samples = []\n        for gt, feat in load(\'cs_fit\', \'gt\', \'feat_b0\'):\n            y = gt[4::8,4::8].ravel()\n            valid = np.flatnonzero(y != IGNORE)\n            idx = rng.choice(valid, min(500, len(valid)), replace=False)\n            samples.append(feat.reshape(256,-1).T[idx].astype(np.float32))\n        x = np.concatenate(samples)\n        # Separate seeded RNG: exact bank saved, not claimed identical to old E3.\n        idx = np.random.default_rng(0).permutation(len(x))[:20000]\n        bank = F.normalize(torch.from_numpy(x[idx]), dim=1).numpy()\n        np.save(reference, bank)\n    bank = torch.from_numpy(np.load(reference)).cuda().float()\n    assert bank.shape == (20000,256)\n    output = {}\n    for domain in DOMAINS:\n        arrays = {\'entropy\':[], \'knn\':[]}\n        count = len(SegSet(domain))\n        for i, (logits, feat) in enumerate(load(domain, \'logits_b0\',\'feat_b0\')):\n            entropy = uncertainty(torch.from_numpy(logits).float())[\'entropy\']\n            entropy = F.avg_pool2d(entropy[None],2)[0].numpy()\n            f = F.normalize(torch.from_numpy(feat.astype(np.float32)).cuda().flatten(1).T, dim=1)\n            distances = []\n            for chunk in f.split(512):\n                distances.append((1-(chunk @ bank.T).topk(10,dim=1).values.mean(1)).cpu())\n            knn = torch.cat(distances).numpy().reshape(64,128)\n            for name, m in [(\'entropy\',entropy),(\'knn\',knn)]:\n                # 8x8 feature cells per 64x64 tile.\n                arrays[name].append(m.reshape(8,8,16,8).mean((1,3)))\n            if (i+1)%10==0 or i+1==count:\n                print(f\'kNN {domain}: {i+1}/{count}\', flush=True)\n        for k,v in arrays.items():\n            output[f\'{domain}__{k}__mean\'] = np.stack(v)\n    path = CACHE / \'e8_tiles_cityscapes.npz\'\n    np.savez(path, **output)\n    write_json(RESULTS/\'knn_config.json\', {\'fit\':\'cs_fit only\', \'bank\':20000, \'k\':10, \'sampling_seed\':0,\n        \'bank_sha256\':hashlib.sha256(reference.read_bytes()).hexdigest(),\n        \'note\':\'Minimal E3 alternative; saved bank differs from historical E3 RNG. Do not compare as exact reproduction.\'})\n    print(\'Сохранено:\', path, flush=True)\n\n@torch.inference_mode()\ndef smoke():\n    from e8_cityscapes import install, predict, token_mask\n    model = load_model(\'b5\'); state = install(model)\n    data = SegSet(\'cs_fit\'); img, _, name = data[0]\n    all_masks = {st: token_mask(np.ones((8,16),bool),st) for st in state}\n    plain = predict(model,state,img,{})\n    masked = predict(model,state,img,all_masks)\n    assert np.array_equal(plain,masked), \'All-true differs from unmasked model\'\n    path = CACHE/\'cs_fit\'/f\'{name}.npz\'\n    assert path.is_file(), \'Сначала выполните E0\'\n    with np.load(path) as z:\n        agreement = float((masked==z[\'pred_b5\']).mean())\n    assert agreement > .999, f\'Cached agreement={agreement}\'\n    selected = np.zeros((8,16),bool); selected[3:5,6:10] = True\n    routed = predict(model,state,img,{2:token_mask(selected,2)})\n    assert routed.shape == SIZE\n    write_json(RESULTS/\'smoke.json\', {\'complete\':True,\'all_true_equal\':True,\'cached_agreement\':agreement,\'selected_mask_forward_ok\':True})\n    print(\'Smoke OK. Совпадение с кэшем:\',agreement, flush=True)\n\n\ndef selftest():\n    from transformers import SegformerConfig, SegformerForSemanticSegmentation\n    from e8_cityscapes import install\n    torch.manual_seed(0)\n    cfg=SegformerConfig(num_channels=3,num_encoder_blocks=4,depths=[1,1,2,1],hidden_sizes=[8,16,32,64],num_attention_heads=[1,2,4,8],sr_ratios=[8,4,2,1],decoder_hidden_size=16,num_labels=19,drop_path_rate=0.)\n    m=SegformerForSemanticSegmentation(cfg).eval()\n    x=torch.randn(1,3,64,128)\n    with torch.inference_mode():\n        original=m(pixel_values=x).logits\n        states=install(m)\n        for st,s in states.items():\n            s.mask=torch.ones(1,(64//(4*2**st))*(128//(4*2**st)),1,dtype=torch.bool)\n        assert torch.equal(original,m(pixel_values=x).logits)\n        outputs=[]; handles=[]\n        for st,s in states.items():\n            s.mask=torch.zeros_like(s.mask)\n            for block in m.segformer.encoder.block[st]:\n                handles.append(block.register_forward_hook(lambda mod,inp,out:outputs.append(torch.equal(out[0],mod.state.entry))))\n        m(pixel_values=x)\n        for h in handles: h.remove()\n        assert outputs and all(outputs)\n    print(\'Tiny SegFormer: all-true / all-false OK\',flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'stage\',choices=[\'cache\',\'scores\',\'smoke\',\'e8\',\'selftest\']);args=p.parse_args()\n    if args.stage!=\'selftest\': assert torch.cuda.is_available(), \'Нужен GPU runtime\'\n    RESULTS.mkdir(parents=True,exist_ok=True);CACHE.mkdir(parents=True,exist_ok=True)\n    import transformers\n    write_json(RESULTS/\'environment.json\', {\'signature\':SIGNATURE,\'python\':platform.python_version(),\'torch\':torch.__version__,\'transformers\':transformers.__version__,\'gpu\':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\'size\':SIZE,\'tile\':TILE,\'domains\':DOMAINS,\'training\':False})\n    if args.stage==\'e8\':\n        from e8_cityscapes import main\n        main()\n    else: globals()[args.stage]()\n'}
for name, text in SOURCES.items():
    path = EXP/name
    if path.exists():
        assert path.read_text() == text, f'Изменён код: {path}. Для новой версии поменяй RUN_NAME.'
    else: path.write_text(text)
code_copy = SAVE/'source'
code_copy.mkdir(exist_ok=True)
for name in SOURCES: shutil.copy2(EXP/name, code_copy/name)
manifest = {'sha256':{k:hashlib.sha256(v.encode()).hexdigest() for k,v in SOURCES.items()},
            'domains':['cs_fit','cs_test'], 'size':[512,1024], 'tile':64,
            'budgets':[0.1,0.2], 'frozen_models':True, 'rank_transductive':True}
manifest_path = SAVE/'run_config.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, 'Конфигурация изменилась: выбери новый RUN_NAME'
else: manifest_path.write_text(json.dumps(manifest, indent=2))
print('Код записан:', EXP)


In [ ]:
# 5. Восстановление после сброса VM и запускатель с логами.
import tarfile, time
for domain in ['cs_fit','cs_test']:
    backup = SAVE/'backups'/f'cache_{domain}.tar'
    if backup.is_file() and not (CACHE/domain).exists():
        print('Восстанавливаем:', backup, flush=True)
        with tarfile.open(backup) as tar:
            members = tar.getmembers()
            for m in members:
                assert m.isfile() or m.isdir()
                assert (CACHE/m.name).resolve().is_relative_to(CACHE.resolve())
            tar.extractall(CACHE)
for name in ['e8_knn_bank.npy','e8_tiles_cityscapes.npz']:
    src = SAVE/name
    if src.is_file() and not (CACHE/name).exists(): shutil.copy2(src,CACHE/name)

def run(stage):
    log_path = SAVE/f'{stage}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('Запуск:', stage, '| Лог:', log_path, flush=True)
    started = time.monotonic()
    with log_path.open('w') as log:
        process = subprocess.Popen([sys.executable,'-u','runner.py',stage],cwd=EXP,
                                   env=os.environ.copy(), stdout=log,stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    status=process.wait(timeout=30)
                    break
                except subprocess.TimeoutExpired:
                    tail = log_path.read_text(errors='replace').splitlines()[-2:]
                    print(f'{stage}: {(time.monotonic()-started)/60:.1f} мин | '+ ' | '.join(tail),flush=True)
        except KeyboardInterrupt:
            process.terminate();process.wait()
            raise
    print(log_path.read_text(errors='replace')[-8000:])
    assert status == 0, f'Ошибка {status}: смотри лог {log_path}'
    print(f'Завершено за {(time.monotonic()-started)/60:.1f} мин')


## Зафиксированные параметры

- Старт: `e9_k4_adapter_v1/adapter_best.pt`, та же нормализация, те же 217 train / 50 validation Frankfurt.
- AdamW, lr=0.0001, weight_decay=0.0001, batch=1, максимум 5 эпох, patience=2, clip_norm=1.
- B5 остаётся в eval, `requires_grad=False`. Маскированный проход **не** обёрнут в inference_mode/no_grad: производные проходят через B5 к предсказанным состояниям.
- Случайные тайловые маски train: 5–30%. Validation: фиксированные воспроизводимые маски 10% и 20% на каждом из 50 кадров.
- Teacher-логиты записываются заранее одним полным проходом на Frankfurt; это около 330 МБ до сжатия. Старые five-state targets переиспользуются; B0 заново для обучения не нужен.

Основной loss на выбранных пикселях рабочей сетки 512×1024:

$$
L = T^2\,\frac{1}{|M|}\sum_{p\in M}
D_{KL}\left(P_{B5}(p;T)\;\Vert\;P_{\mathrm{masked}}(p;T)\right)
+0.01 L_{\mathrm{states}},\qquad T=2.
$$

`L_states` — прежняя стандартизованная MSE: половина веса четырём контекстам вместе и половина final. Логиты интерполируются к рабочему разрешению перед KL, поэтому loss включает границы выбранных регионов. GT не участвует в этих потерях; KL не исключает ignore-пиксели, поскольку учитель определён и там. GT используется для итоговой оценки mIoU.

**Эпоха 0 допускается как лучший checkpoint.** Если новая validation-потеря не улучшится, итогом останутся исходные веса. Критерий выбора — только validation KL+0.01MSE, не cs_test mIoU.


In [ ]:
# 6. Исходные результаты и отдельная папка нового опыта.
BASE=SAVE/'e9_k4_adapter_v1'
for p in [BASE/'adapter_best.pt',BASE/'normalization.npz',BASE/'config.json',BASE/'splits.json',
          BASE/'training_complete.json',BASE/'cs_test_progress.npz',
          SAVE/'e8c_k_sweep_v1/cs_test_progress.npz',SAVE/'e8c_k_sweep_v1/cost_scenarios.json',
          SAVE/'e8c_k_sweep_v1/cost_measured.json',CACHE/'e8_tiles_cityscapes.npz']:
    assert p.is_file(), f'Не найден {p}; восстанови предыдущий запуск.'
assert (BASE/'targets/cs_fit').is_dir(), 'Нужна папка five-state targets E9-k4 на Drive'
OUT=SAVE/'e12_output_distill_v1'
LOCAL=WORK/'e12_output_distill_v1'
OUT.mkdir(exist_ok=True);LOCAL.mkdir(exist_ok=True)
os.environ.update(E12_OUT=str(OUT),E12_LOCAL=str(LOCAL),E12_BASE=str(BASE))
print('Новый опыт:',OUT)


In [ ]:
# 7. Код встроен в notebook. Исходный E9-k4 не меняется.
NEW_SOURCES={'e9k_base_eval.py': '"""E9-k4: B0 F3 -> four midpoint contexts plus final. Teacher-assisted diagnostic."""\nimport os\nos.environ[\'USE_TF\']=\'0\'\nimport argparse, csv, hashlib, json, time, shutil\nfrom pathlib import Path\nfrom types import SimpleNamespace\nfrom collections import Counter\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader\nimport torch.nn.functional as F\nfrom common import CACHE, RESULTS, SIZE, TILE, SegSet, load_model, confusion, miou, top_tiles\n\nOUT=Path(os.environ[\'E9K_OUT\'])\nLOCAL=Path(os.environ[\'E9K_LOCAL\'])\nPREVIOUS=Path(os.environ.get(\'E9K_PREVIOUS\',str(RESULTS.parent/\'e9_adapter_v1\')))\nSTATES=[\'s5\',\'s15\',\'s25\',\'s35\',\'final\']\nCONTEXTS=STATES[:-1]\nFEATURE_MODELS={}\nCONFIG={\'schema\':\'e9-k4-five-state-adapter-v1\',\'seed\':0,\'input\':\'B0 normalized encoder F3 (stage index 2 output)\',\n        \'input_channels\':160,\'target_channels\':320,\'hidden\':128,\'target_states\':STATES,\n        \'epochs\':20,\'patience\':4,\'lr\':.001,\'weight_decay\':.0001,\'batch\':2,\n        \'budgets\':[.1,.2],\'evaluation_domains\':[\'cs_test\'],\'methods\':[\'rank\'],\n        \'teacher_assisted\':True,\'keep_real_b5_early_stages\':True,\'routed_entry\':\'real B5\',\n        \'loss\':\'0.5 mean normalized MSE over four contexts + 0.5 normalized final MSE\',\n        \'fit_domains\':[\'cs_fit\'],\'size\':[512,1024], \'compute_benchmark\':False}\n\n\ndef write_json(path,data):\n    path.parent.mkdir(parents=True,exist_ok=True)\n    p=path.with_suffix(path.suffix+\'.part\');p.write_text(json.dumps(data,indent=2));p.replace(path)\n\ndef atomic_torch(path,data):\n    p=path.with_suffix(\'.pt.part\');torch.save(data,p);p.replace(path)\n\ndef sha(path):return hashlib.sha256(path.read_bytes()).hexdigest()\n\ndef initialize():\n    OUT.mkdir(parents=True,exist_ok=True);LOCAL.mkdir(parents=True,exist_ok=True)\n    cfg=dict(CONFIG,source_sha256=sha(Path(__file__)),\n             common_sha256=sha(Path(__file__).parent/\'common.py\'),\n             tile_features_sha256=sha(CACHE/\'e8_tiles_cityscapes.npz\'),\n             previous_split_sha256=sha(PREVIOUS/\'splits.json\'))\n    if (PREVIOUS/\'config.json\').exists():\n        old=json.loads((PREVIOUS/\'config.json\').read_text())\n        assert old[\'common_sha256\']==cfg[\'common_sha256\'] and old[\'tile_features_sha256\']==cfg[\'tile_features_sha256\'], \'Previous E9 used different preprocessing or tile features\'\n    p=OUT/\'config.json\'\n    if p.exists():assert json.loads(p.read_text())==cfg,\'Configuration differs: use a new E9_OUT\'\n    else:write_json(p,cfg)\n    return cfg\n\n\nclass ContextBlock(nn.Module):\n    def __init__(self,layer,state,index,n):\n        super().__init__();self.layer=layer;self.state=state;self.index=index;self.n=n\n    def forward(self,x,*args,**kwargs):\n        s=self.state\n        if s.recording:\n            if self.index in [5,15,25,35]:s.real[\'s\'+str(self.index)]=x\n            out=self.layer(x,*args,**kwargs)\n            if self.index==self.n-1:s.real[\'final\']=out[0]\n            return out\n        if s.mask is None:return self.layer(x,*args,**kwargs)\n        ctx=s.contexts[CONTEXTS[self.index*4//self.n]]\n        out=self.layer(torch.where(s.mask,x,ctx),*args,**kwargs)\n        y=torch.where(s.mask,out[0],ctx)\n        if self.index==self.n-1:y=torch.where(s.mask,y,s.output)\n        return (y,)+tuple(out[1:])\n\n\ndef install(model):\n    state=SimpleNamespace(recording=False,real={},mask=None,contexts=None,output=None)\n    layers=model.segformer.encoder.block[2]\n    assert len(layers)==40,\'Expected B5 stage3 with 40 blocks\'\n    for i,layer in enumerate(layers):layers[i]=ContextBlock(layer,state,i,len(layers))\n    return state\n\n@torch.inference_mode()\ndef forward(model,state,img,recording=False,mask=None,contexts=None,output=None):\n    state.recording=recording;state.mask=mask;state.contexts=contexts;state.output=output\n    try:return model(pixel_values=img).logits\n    finally:state.recording=False;state.mask=None;state.contexts=None;state.output=None\n\n\ndef prediction(logits):\n    return F.interpolate(logits.float(),SIZE,mode=\'bilinear\',align_corners=False).argmax(1)[0].byte().cpu().numpy()\n\ndef target_path(domain,name):return LOCAL/\'targets\'/domain/f\'{name}.npz\'\n\ndef check_target(path):\n    if not path.is_file():return False\n    with np.load(path,allow_pickle=False) as z:\n        assert z[\'x\'].shape==(160,32,64)\n        assert all(z[k].shape==(320,32,64) for k in STATES)\n        assert all(np.isfinite(z[k]).all() for k in [\'x\']+STATES)\n    return True\n\n\ndef split_ids(names):\n    # City+sequence, not neighboring-frame random split.\n    groups={n:\'_\'.join(n.split(\'_\')[:2]) for n in names}\n    counts=Counter(groups.values())\n    assert len(counts)>=2,\'Need at least two Frankfurt sequences\'\n    val_group=min(counts,key=lambda g:(counts[g],g))\n    train=[n for n in names if groups[n]!=val_group]\n    val=[n for n in names if groups[n]==val_group]\n    assert train and val and len(val)<len(train),(counts, val_group)\n    return {\'train\':train,\'val\':val,\'val_sequence\':val_group,\'sequence_counts\':dict(counts),\n            \'note\':\'Validation is smallest Frankfurt sequence; cs_test never used for selection.\'}\n\n@torch.inference_mode()\ndef feature(domain,name,img=None,device=\'cuda\'):\n    p=LOCAL/\'b0_features\'/domain/(name+\'.npy\')\n    if p.is_file():\n        a=np.load(p,allow_pickle=False)\n    else:\n        candidates=[PREVIOUS/\'targets\'/domain/(name+\'.npz\'),\n                    LOCAL.parent/\'e9_adapter_v1/targets\'/domain/(name+\'.npz\')]\n        source=next((x for x in candidates if x.is_file()),None)\n        if source:\n            with np.load(source,allow_pickle=False) as z:a=z[\'x\']\n        else:\n            assert img is not None,\'Need image to regenerate missing B0 F3\'\n            if device not in FEATURE_MODELS:\n                FEATURE_MODELS[device]=load_model(\'b0\',device=device).requires_grad_(False)\n            a=FEATURE_MODELS[device](pixel_values=img,output_hidden_states=True).hidden_states[2][0].cpu().numpy()\n        p.parent.mkdir(parents=True,exist_ok=True)\n        temp=p.with_suffix(\'.npy.part\')\n        with temp.open(\'wb\') as f:np.save(f,a)\n        temp.replace(p)\n    assert a.shape==(160,32,64) and np.isfinite(a).all()\n    return a\n\n\ndef normalized_loss(pred,target):\n    assert pred.shape[1]==target.shape[1]==1600\n    return .5*F.mse_loss(pred[:,:1280],target[:,:1280])+.5*F.mse_loss(pred[:,1280:],target[:,1280:])\n\n@torch.inference_mode()\ndef export_targets():\n    b5=load_model(\'b5\').requires_grad_(False);state=install(b5)\n    assert len(b5.segformer.encoder.block[2])==40,\'Expected B5 stage 3 depth=40\'\n    for domain in [\'cs_fit\']:\n        data=SegSet(domain)\n        expected=267 if domain==\'cs_fit\' else 233\n        assert len(data)==expected\n        assert [p.stem for p,_ in data.items]==[p.stem for p in sorted((CACHE/domain).glob(\'*.npz\'))]\n        start=time.monotonic()\n        for i,(image,gt_path) in enumerate(data.items):\n            name=image.stem;p=target_path(domain,name)\n            backup=OUT/\'targets\'/domain/p.name\n            p.parent.mkdir(parents=True,exist_ok=True);backup.parent.mkdir(parents=True,exist_ok=True)\n            if not p.exists() and backup.exists():shutil.copy2(backup,p)\n            if not check_target(p):\n                img,_,_=data[i];x=img[None].cuda().half()\n                f3=feature(domain,name,x,\'cuda\')\n                pred=prediction(forward(b5,state,x,recording=True))\n                with np.load(CACHE/domain/p.name) as z:agree=float((pred==z[\'pred_b5\']).mean())\n                assert agree>.999,f\'B5/cache mismatch {name}: {agree}\'\n                packed={\'x\':f3}\n                for k in STATES:\n                    v=state.real[k]\n                    assert v.shape==(1,2048,320),v.shape\n                    packed[k]=v[0].T.reshape(320,32,64).cpu().numpy()\n                tmp=p.with_suffix(\'.npz.part\')\n                with tmp.open(\'wb\') as f:np.savez_compressed(f,**packed)\n                tmp.replace(p);state.real={}\n            if not backup.exists():\n                tmp=backup.with_suffix(\'.npz.part\');shutil.copy2(p,tmp);tmp.replace(backup)\n            if (i+1)%10==0 or i+1==len(data):\n                print(f\'Targets {domain}: {i+1}/{len(data)}, {(time.monotonic()-start)/60:.1f} мин\',flush=True)\n    names=[p.stem for p,_ in SegSet(\'cs_fit\').items]\n    expected=split_ids(names)\n    splits=json.loads((PREVIOUS/\'splits.json\').read_text())\n    assert splits[\'train\']==expected[\'train\'] and splits[\'val\']==expected[\'val\'], \'Old split differs\'\n    p=OUT/\'splits.json\'\n    if p.exists():assert json.loads(p.read_text())==splits\n    else:write_json(p,splits)\n    print(\'Train:\',len(splits[\'train\']),\'val:\',len(splits[\'val\']),\'sequence:\',splits[\'val_sequence\'],flush=True)\n    write_json(OUT/\'targets_complete.json\',{\'complete\':True,\'domains\':{\'cs_fit\':267},\'cs_test_targets_exported\':False})\n\n\ndef ensure_local(domain,names):\n    for n in names:\n        p=target_path(domain,n)\n        if not p.exists():\n            p.parent.mkdir(parents=True,exist_ok=True)\n            shutil.copy2(OUT/\'targets\'/domain/p.name,p)\n        check_target(p)\n\n\ndef normalization(names):\n    p=OUT/\'normalization.npz\'\n    if p.exists():\n        with np.load(p) as z:return {k:z[k] for k in z.files}\n    sums={k:np.zeros(d,np.float64) for k,d in [(\'x\',160)]+[(name,320) for name in STATES]}\n    squares={k:v.copy() for k,v in sums.items()};n=0\n    for name in names:\n        with np.load(target_path(\'cs_fit\',name)) as z:\n            for k in sums:\n                a=z[k].astype(np.float64).reshape(len(sums[k]),-1)\n                sums[k]+=a.sum(1);squares[k]+=(a*a).sum(1)\n            n+=a.shape[1]\n    result={}\n    for k in sums:\n        mean=sums[k]/n\n        std=np.sqrt(np.maximum(squares[k]/n-mean**2,1e-8))\n        result[k+\'_mean\']=mean.astype(np.float32)\n        result[k+\'_std\']=std.astype(np.float32)\n    np.savez(p,**result)\n    return result\n\nclass Adapter(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.net=nn.Sequential(nn.Conv2d(160,128,1),nn.GELU(),nn.Conv2d(128,1600,1))\n    def forward(self,x):return self.net(x)\n\nclass Maps(Dataset):\n    def __init__(self,names,norm):self.names=names;self.norm=norm\n    def __len__(self):return len(self.names)\n    def __getitem__(self,i):\n        with np.load(target_path(\'cs_fit\',self.names[i])) as z:\n            def standard(k):return (z[k].astype(np.float32)-self.norm[k+\'_mean\'][:,None,None])/self.norm[k+\'_std\'][:,None,None]\n            return torch.from_numpy(standard(\'x\')),torch.from_numpy(np.concatenate([standard(k) for k in STATES]))\n\n@torch.no_grad()\ndef val_loss(model,loader,device):\n    model.eval();s=0.;n=0\n    for x,y in loader:\n        x=x.to(device);y=y.to(device)\n        s+=normalized_loss(model(x),y).item()*len(x);n+=len(x)\n    return s/n\n\n\ndef train(device="cuda"):\n    torch.manual_seed(0);np.random.seed(0)\n    splits=json.loads((OUT/\'splits.json\').read_text())\n    ensure_local(\'cs_fit\',splits[\'train\']+splits[\'val\'])\n    norm=normalization(splits[\'train\'])\n    done=OUT/\'training_complete.json\'\n    if done.exists():\n        d=json.loads(done.read_text())\n        assert d[\'checkpoint_sha256\']==sha(OUT/\'adapter_best.pt\')\n        print(\'Адаптер уже обучен, пропускаем.\',flush=True);return\n    model=Adapter().to(device)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=CONFIG[\'lr\'],weight_decay=CONFIG[\'weight_decay\'])\n    val_loader=DataLoader(Maps(splits[\'val\'],norm),batch_size=2,num_workers=0)\n    last=OUT/\'adapter_last.pt\';best=float(\'inf\');bad=0;history=[];start=0;best_epoch=None\n    if last.exists():\n        checkpoint=torch.load(last,map_location=device,weights_only=True)\n        model.load_state_dict(checkpoint[\'model\']);optimizer.load_state_dict(checkpoint[\'optimizer\'])\n        best=checkpoint[\'best\'];bad=checkpoint[\'bad\'];history=checkpoint[\'history\'];start=checkpoint[\'epoch\']\n        best_epoch=checkpoint[\'best_epoch\']\n        print(\'Возобновляем после эпохи\',start,flush=True)\n    for epoch in range(start,CONFIG[\'epochs\']):\n        if bad>=CONFIG[\'patience\']:break\n        generator=torch.Generator().manual_seed(CONFIG[\'seed\']+epoch)\n        loader=DataLoader(Maps(splits[\'train\'],norm),batch_size=CONFIG[\'batch\'],shuffle=True,generator=generator,num_workers=0)\n        model.train();total=0.;count=0\n        for x,y in loader:\n            x=x.to(device);y=y.to(device);optimizer.zero_grad(set_to_none=True)\n            pred=model(x);loss=normalized_loss(pred,y)\n            assert torch.isfinite(loss),\'Nonfinite loss\'\n            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),5.)\n            optimizer.step();total+=loss.item()*len(x);count+=len(x)\n        val=val_loss(model,val_loader,device)\n        row={\'epoch\':epoch+1,\'train_weighted_normalized_mse\':total/count,\'val_weighted_normalized_mse\':val}\n        history.append(row)\n        if val<best:\n            best=val;bad=0;best_epoch=epoch+1\n            atomic_torch(OUT/\'adapter_best.pt\',{\'model\':model.state_dict(),\'epoch\':best_epoch,\'val_mse\':best})\n        else:bad+=1\n        atomic_torch(last,{\'model\':model.state_dict(),\'optimizer\':optimizer.state_dict(),\'epoch\':epoch+1,\n                          \'best\':best,\'bad\':bad,\'best_epoch\':best_epoch,\'history\':history})\n        write_json(OUT/\'training_history.json\',history)\n        print(row,flush=True)\n    best_checkpoint=torch.load(OUT/\'adapter_best.pt\',map_location=device,weights_only=True)\n    model.load_state_dict(best_checkpoint[\'model\']);model.eval()\n    diag={k:{\'error\':0.,\'energy\':0.,\'mean_error\':0.} for k in STATES}\n    with torch.no_grad():\n        for name in splits[\'val\']:\n            with np.load(target_path(\'cs_fit\',name)) as z:\n                x=(z[\'x\'].astype(np.float32)-norm[\'x_mean\'][:,None,None])/norm[\'x_std\'][:,None,None]\n                y=model(torch.from_numpy(x)[None].to(device))[0].cpu().numpy()\n                for i,k in enumerate(STATES):\n                    target=z[k].astype(np.float64)\n                    output=y[i*320:(i+1)*320]*norm[k+\'_std\'][:,None,None]+norm[k+\'_mean\'][:,None,None]\n                    diag[k][\'error\']+=float(((output-target)**2).sum())\n                    diag[k][\'energy\']+=float((target**2).sum())\n                    diag[k][\'mean_error\']+=float(((norm[k+\'_mean\'][:,None,None]-target)**2).sum())\n    diagnostics={k:{\'relative_squared_error\':d[\'error\']/max(d[\'energy\'],1e-12),\n                    \'error_vs_train_mean_baseline\':d[\'error\']/max(d[\'mean_error\'],1e-12)} for k,d in diag.items()}\n    write_json(OUT/\'adapter_validation.json\',diagnostics)\n    summary={\'complete\':True,\'best_epoch\':best_epoch,\'val_weighted_normalized_mse\':best,\n             \'parameters\':sum(p.numel() for p in model.parameters()),\'train_frames\':len(splits[\'train\']),\n             \'val_frames\':len(splits[\'val\']),\'checkpoint_sha256\':sha(OUT/\'adapter_best.pt\'),\n             \'normalization_sha256\':sha(OUT/\'normalization.npz\')}\n    write_json(done,summary);print(summary,diagnostics,flush=True)\n\n@torch.inference_mode()\ndef estimate(model,norm,array,device):\n    x=(array.astype(np.float32)-norm[\'x_mean\'][:,None,None])/norm[\'x_std\'][:,None,None]\n    y=model(torch.from_numpy(x)[None].to(device))[0]\n    result={}\n    for i,k in enumerate(STATES):\n        output=y[i*320:(i+1)*320]\n        output=output*torch.as_tensor(norm[k+\'_std\'],device=device)[:,None,None]+torch.as_tensor(norm[k+\'_mean\'],device=device)[:,None,None]\n        result[k]=output.flatten(1).T[None].to(dtype=torch.float16 if str(device).startswith(\'cuda\') else torch.float32)\n    return result\n\n\ndef rank(x):return x.ravel().argsort().argsort().reshape(x.shape)\n\n\ndef csv_rows(path,rows):\n    if not rows:return\n    fields=list(dict.fromkeys(k for r in rows for k in r))\n    with path.open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows(rows)\n\n\ndef frame_controls():\n    """Exact floor/ceil counts around the *arithmetic* k4 budget; no linear mIoU interpolation."""\n    cost_root=RESULTS.parent/\'e8c_k_sweep_v1\'\n    measured=json.loads((cost_root/\'cost_measured.json\').read_text())\n    scenarios=json.loads((cost_root/\'cost_scenarios.json\').read_text())[\'rows\']\n    b0_cost=measured[\'models\'][\'b0\'][\'fvcore\'][\'counted_total_gops\']\n    b5_cost=measured[\'models\'][\'b5\'][\'fvcore\'][\'counted_total_gops\']\n    data=SegSet(\'cs_test\');names=[p.stem for p,_ in data.items];n=len(names)\n    assert names==[p.stem for p in sorted((CACHE/\'cs_test\').glob(\'*.npz\'))]\n    with np.load(CACHE/\'e8_tiles_cityscapes.npz\') as z:\n        entropy=z[\'cs_test__entropy__mean\'];knn=z[\'cs_test__knn__mean\']\n    scores={\'raw_entropy\':entropy.mean((1,2)),\'rank_mean\':(rank(entropy)+rank(knn)).mean((1,2))}\n    conf=np.zeros((n,2,19,19),np.int64)\n    for i,name in enumerate(names):\n        with np.load(CACHE/\'cs_test\'/(name+\'.npz\')) as z:\n            conf[i,0]=confusion(z[\'pred_b0\'],z[\'gt\']);conf[i,1]=confusion(z[\'pred_b5\'],z[\'gt\'])\n    rows=[];per_frame=[];labels=[]\n    for b in CONFIG[\'budgets\']:\n        scenario=next(r for r in scenarios if r[\'k\']==4 and r[\'budget\']==b)\n        assert scenario[\'adapter_parameters_scenario\']==227008\n        target=float(scenario[\'hybrid_planning_count_halo_aware\'])\n        fraction=(target-b0_cost)/b5_cost\n        assert 0<=fraction<=1\n        raw=fraction*n\n        counts=[(\'floor\',int(np.floor(raw))),(\'ceil\',int(np.ceil(raw)))]\n        for method,score in scores.items():\n            order=np.argsort(-score,kind=\'stable\')\n            for side,count in counts:\n                mask=np.zeros(n,bool);mask[order[:count]]=True\n                matrices=np.where(mask[:,None,None],conf[:,1],conf[:,0])\n                actual=b0_cost+float(mask.mean())*b5_cost\n                label=f\'{method}_b{b}_{side}\'\n                rows.append({\'label\':label,\'tile_budget_reference\':b,\'method\':method,\'bracket\':side,\n                             \'target_frame_fraction\':fraction,\'actual_frame_fraction\':float(mask.mean()),\n                             \'selected_frames\':count,\'cached_miou\':float(miou(matrices.sum(0))),\n                             \'target_hybrid_count\':target,\'actual_counted_gops\':actual,\n                             \'count_difference\':actual-target,\n                             \'scope\':\'arithmetic-budget bracket, not matched measured latency\'})\n                per_frame.append(matrices);labels.append(label)\n    np.savez_compressed(OUT/\'frame_controls.npz\',confs=np.stack(per_frame,axis=1),names=np.array(names),labels=np.array(labels))\n    write_json(OUT/\'frame_controls.json\',{\'rows\':rows,\'cost_source\':str(cost_root),\n        \'cost_source_sha256\':sha(cost_root/\'cost_scenarios.json\'),\n        \'warning\':\'The ROI target is an unimplemented hybrid arithmetic scenario. Whole-frame quality is cached. This is not a measured equal-time comparison.\'})\n    csv_rows(OUT/\'frame_controls.csv\',rows)\n    for row in rows:print(row,flush=True)\n\n\ndef reference_k4(domain):\n    path=RESULTS.parent/\'e8c_k_sweep_v1/e8c_quality.json\'\n    rows=json.loads(path.read_text())[domain][\'rows\']\n    refs={float(r[\'budget\']):float(r[\'miou\']) for r in rows if r[\'variant\']==\'k4\'}\n    assert all(b in refs for b in CONFIG[\'budgets\'])\n    return refs\n\n\ndef save_eval_progress(path,completed,confs,agreements,names,modes,model_sha):\n    p=path.with_suffix(\'.npz.part\')\n    with p.open(\'wb\') as f:\n        np.savez_compressed(f,completed=np.array(completed),confs=confs[:completed],agreements=agreements,\n                            names=np.array(names),modes=np.array([\'b0\',\'full_tiles\']+modes),\n                            budgets=np.array(CONFIG[\'budgets\']),checkpoint_sha256=np.array(model_sha))\n    p.replace(path)\n\n@torch.inference_mode()\ndef evaluate(device=\'cuda\'):\n    summary=json.loads((OUT/\'training_complete.json\').read_text());model_sha=sha(OUT/\'adapter_best.pt\')\n    assert summary[\'checkpoint_sha256\']==model_sha\n    assert summary[\'normalization_sha256\']==sha(OUT/\'normalization.npz\')\n    report_path=OUT/\'routing.json\'\n    report=json.loads(report_path.read_text()) if report_path.exists() else {\'checkpoint_sha256\':model_sha}\n    assert report[\'checkpoint_sha256\']==model_sha,\'Checkpoint changed: use new output directory\'\n    adapter=Adapter().to(device).eval().requires_grad_(False)\n    adapter.load_state_dict(torch.load(OUT/\'adapter_best.pt\',map_location=device,weights_only=True)[\'model\'])\n    with np.load(OUT/\'normalization.npz\') as z:norm={k:z[k] for k in z.files}\n    model=load_model(\'b5\',device=device).requires_grad_(False)\n    state=install(model);dtype=torch.float16 if str(device).startswith(\'cuda\') else torch.float32\n    modes=[\'true_context_true_output\',\'pred_context_true_output\',\'true_context_pred_output\',\n           \'pred_context_pred_output\',\'mean_context_mean_output\']\n    previous_path=PREVIOUS/\'e9_routing.json\'\n    previous=json.loads(previous_path.read_text()) if previous_path.exists() else {}\n    with np.load(CACHE/\'e8_tiles_cityscapes.npz\') as z:feats={k:z[k] for k in z.files}\n    for domain in CONFIG[\'evaluation_domains\']:\n        if domain in report:\n            print(domain,\'готов, пропускаем\',flush=True);continue\n        refs=reference_k4(domain)\n        data=SegSet(domain);names=[p.stem for p,_ in data.items];n=len(names)\n        assert names==[p.stem for p in sorted((CACHE/domain).glob(\'*.npz\'))]\n        score=rank(feats[f\'{domain}__entropy__mean\'])+rank(feats[f\'{domain}__knn__mean\'])\n        masks=[top_tiles(score,b) for b in CONFIG[\'budgets\']]\n        confs=np.zeros((n,len(masks),len(modes)+2,19,19),np.int64)\n        agreements=np.zeros((len(masks),len(modes)+1,2),np.int64)\n        progress=OUT/f\'{domain}_progress.npz\';done=0\n        if progress.exists():\n            with np.load(progress,allow_pickle=False) as z:\n                assert str(z[\'checkpoint_sha256\'])==model_sha\n                assert list(z[\'names\'])==names and list(z[\'modes\'])==[\'b0\',\'full_tiles\']+modes\n                assert np.array_equal(z[\'budgets\'],CONFIG[\'budgets\'])\n                done=int(z[\'completed\']);assert 0<=done<=n\n                confs[:done]=z[\'confs\'];agreements[:]=z[\'agreements\']\n            print(\'Продолжаем оценку после\',done,\'кадров\',flush=True)\n        mean={k:torch.as_tensor(norm[k+\'_mean\'],device=device)[None,None,:].expand(1,2048,320).to(dtype=dtype) for k in STATES}\n        start=time.monotonic()\n        for i in range(done,n):\n            name=names[i];img,_,_=data[i];img=img[None].to(device=device,dtype=dtype)\n            full_logits=forward(model,state,img,recording=True);full_pred=prediction(full_logits)\n            with np.load(CACHE/domain/(name+\'.npz\')) as z:gt=z[\'gt\'];b0=z[\'pred_b0\'];b5=z[\'pred_b5\']\n            assert (full_pred==b5).mean()>.999,f\'B5/cache mismatch {name}\'\n            pred=estimate(adapter,norm,feature(domain,name,img,device),device)\n            true=state.real\n            options=[(true,true[\'final\']),(pred,true[\'final\']),(true,pred[\'final\']),\n                     (pred,pred[\'final\']),(mean,mean[\'final\'])]\n            if i==done:\n                ones=torch.ones((1,2048,1),device=device,dtype=torch.bool)\n                check=forward(model,state,img,mask=ones,contexts=pred,output=pred[\'final\'])\n                assert torch.equal(check,full_logits),\'All-true changed logits\'\n                write_json(OUT/\'sanity.json\',{\'all_true_predicted_states_equal\':True,\'first_checked_image\':name})\n            for j,m in enumerate(masks):\n                px=m[i].repeat(TILE,0).repeat(TILE,1)\n                tm=torch.from_numpy(m[i]).to(device).repeat_interleave(4,0).repeat_interleave(4,1).reshape(1,2048,1)\n                confs[i,j,0]=confusion(b0,gt);confs[i,j,1]=confusion(np.where(px,b5,b0),gt)\n                agreements[j,0]+=[int((b0==b5)[px].sum()),int(px.sum())]\n                for k,(contexts,output) in enumerate(options):\n                    modified=prediction(forward(model,state,img,mask=tm,contexts=contexts,output=output))\n                    confs[i,j,k+2]=confusion(np.where(px,modified,b0),gt)\n                    agreements[j,k+1]+=[int((modified==b5)[px].sum()),int(px.sum())]\n            state.real={}\n            if (i+1)%10==0 or i+1==n:save_eval_progress(progress,i+1,confs,agreements,names,modes,model_sha)\n            if (i+1)%5==0 or i+1==n:\n                elapsed=time.monotonic()-start\n                print(f\'E9-k4 {domain}: {i+1}/{n}, {elapsed/60:.1f} мин, ETA ~{elapsed/(i+1-done)*(n-i-1)/60:.1f} мин\',flush=True)\n        checks=[{\'budget\':b,\'delta_vs_k4_pp\':float(miou(confs[:,j,2].sum(0)))-refs[b]} for j,b in enumerate(CONFIG[\'budgets\'])]\n        write_json(OUT/\'reference_check.json\',checks)\n        assert all(abs(r[\'delta_vs_k4_pp\'])<.01 for r in checks),\'Teacher k4 control differs from E8c\'\n        rows=[]\n        for j,b in enumerate(CONFIG[\'budgets\']):\n            values=[float(miou(v)) for v in confs[:,j].sum(0)]\n            old=next((r[\'miou\'] for r in previous.get(domain,[]) if r[\'budget\']==b and r[\'variant\']==\'pred_context_pred_output\'),None)\n            for k,mode in enumerate(modes):\n                a=agreements[j,k+1]\n                row={\'domain\':domain,\'budget\':b,\'variant\':mode,\'miou\':values[k+2],\n                     \'delta_vs_b0_pp\':values[k+2]-values[0],\n                     \'delta_vs_true_k4_pp\':values[k+2]-values[2],\n                     \'delta_vs_full_tiles_pp\':values[k+2]-values[1],\n                     \'agree_b5_percent\':100*int(a[0])/max(int(a[1]),1),\n                     \'previous_e9_pred_both_miou\':old}\n                if old is not None:row[\'delta_vs_previous_e9_pred_both_pp\']=values[k+2]-old\n                rows.append(row);print(row,flush=True)\n        report[domain]=rows;write_json(report_path,report)\n    csv_rows(OUT/\'routing.csv\',[r for d in CONFIG[\'evaluation_domains\'] for r in report[d]])\n\n\ndef bootstrap():\n    report=json.loads((OUT/\'routing.json\').read_text())\n    with np.load(OUT/\'frame_controls.npz\') as z:frame_confs=z[\'confs\'];frame_names=z[\'names\'];labels=list(z[\'labels\'])\n    frame_meta=json.loads((OUT/\'frame_controls.json\').read_text())[\'rows\']\n    old_path=PREVIOUS/\'cs_test_frame_confusions.npz\'\n    old=None\n    if old_path.exists():\n        with np.load(old_path) as z:old={k:z[k] for k in z.files}\n    rng=np.random.default_rng(0);rows=[]\n    for domain in CONFIG[\'evaluation_domains\']:\n        assert domain in report\n        with np.load(OUT/f\'{domain}_progress.npz\') as z:\n            conf=z[\'confs\'];names=z[\'names\'];modes=list(z[\'modes\']);budgets=z[\'budgets\']\n            assert len(conf)==len(names)==int(z[\'completed\'])\n        assert np.array_equal(names,frame_names)\n        for unit in [\'frame\',\'sequence\']:\n            ids=np.arange(len(names)).astype(str) if unit==\'frame\' else np.array([\'_\'.join(n.split(\'_\')[:2]) for n in names])\n            unique=sorted(set(ids))\n            def group(x):return np.stack([x[ids==g].sum(0) for g in unique])\n            for j,b in enumerate(budgets):\n                ours=conf[:,j,modes.index(\'pred_context_pred_output\')]\n                references={mode:conf[:,j,modes.index(mode)] for mode in [\'true_context_true_output\',\'mean_context_mean_output\',\'full_tiles\']}\n                for item in frame_meta:\n                    if item[\'tile_budget_reference\']==b:references[\'frame_\'+item[\'label\']]=frame_confs[:,labels.index(item[\'label\'])]\n                if old is not None:\n                    assert np.array_equal(old[\'names\'],names)\n                    oj=list(old[\'budgets\']).index(b);ok=list(old[\'modes\']).index(\'pred_context_pred_output\')\n                    references[\'previous_e9_pred_both\']=old[\'confs\'][:,oj,ok]\n                a=group(ours)\n                for label,reference in references.items():\n                    other=group(reference);diff=[]\n                    for _ in range(1000):\n                        idx=rng.integers(0,len(unique),len(unique))\n                        diff.append(miou(a[idx].sum(0))-miou(other[idx].sum(0)))\n                    ci=np.percentile(diff,[2.5,97.5])\n                    rows.append({\'domain\':domain,\'budget\':float(b),\'unit\':unit,\'groups\':len(unique),\n                                 \'comparison\':\'pred_all minus \'+label,\n                                 \'delta_miou_pp\':float(miou(a.sum(0))-miou(other.sum(0))),\n                                 \'ci_low\':float(ci[0]),\'ci_high\':float(ci[1]),\n                                 \'scope\':\'paired quality comparison; frame controls matched by arithmetic proxy only\'})\n    write_json(OUT/\'bootstrap.json\',rows);csv_rows(OUT/\'bootstrap.csv\',rows)\n    for row in rows:print(row,flush=True)\n\n\ndef main():\n    parser=argparse.ArgumentParser();parser.add_argument(\'stage\',choices=[\'frame_controls\',\'export_targets\',\'train\',\'evaluate\',\'bootstrap\'])\n    args=parser.parse_args();initialize()\n    if args.stage not in [\'frame_controls\',\'bootstrap\']:assert torch.cuda.is_available(),\'Need GPU\'\n    import transformers,platform\n    write_json(OUT/\'environment.json\',{\'python\':platform.python_version(),\'torch\':torch.__version__,\n        \'transformers\':transformers.__version__,\'gpu\':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})\n    globals()[args.stage]()\nif __name__==\'__main__\':main()\n', 'e12_output_distill.py': '"""Bounded adapter-only output distillation. Frozen B5, explicit checkpoint tensor inputs."""\nimport os\nos.environ[\'USE_TF\']=\'0\'\nimport argparse,hashlib,json,time,csv,shutil\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport numpy as np\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\nfrom common import CACHE,RESULTS,SIZE,TILE,SegSet,load_model,confusion,miou,top_tiles\n\nOUT=Path(os.environ[\'E12_OUT\']);LOCAL=Path(os.environ[\'E12_LOCAL\'])\nBASE=Path(os.environ[\'E12_BASE\'])\nos.environ[\'E9K_OUT\']=str(OUT);os.environ[\'E9K_LOCAL\']=str(LOCAL)\nos.environ[\'E9K_PREVIOUS\']=str(RESULTS.parent/\'e9_adapter_v1\')\nimport e9k_base_eval as base\nSTATES=base.STATES;CONTEXTS=base.CONTEXTS\nCONFIG={\'schema\':\'e12-output-distill-v1\',\'seed\':0,\'epochs\':5,\'patience\':2,\'batch\':1,\n        \'lr\':1e-4,\'weight_decay\':1e-4,\'temperature\':2.,\'state_weight\':.01,\n        \'train_mask_fraction\':[.05,.30],\'val_mask_fractions\':[.1,.2],\n        \'clip_norm\':1.,\'amp_initial_scale\':1024.,\'checkpointing\':\'non-reentrant; explicit context/output/mask tensors\',\n        \'teacher_assisted\':True,\'b0_frozen\':True,\'b5_frozen_eval\':True,\n        \'loss\':\'T^2 KL(full B5 teacher || masked student) on selected full-resolution pixels + .01 balanced state MSE\',\n        \'selection\':\'fixed validation loss, epoch0 baseline eligible\',\n        \'gate\':\'training only if true-k4 reserve lower nominal 95% CI >0 at at least one hypothetical E11 budget; exploratory\'}\n\ndef write_json(p,data):\n    p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_suffix(p.suffix+\'.part\')\n    tmp.write_text(json.dumps(data,indent=2));tmp.replace(p)\n\ndef atomic_torch(p,data):\n    tmp=p.with_suffix(\'.pt.part\');torch.save(data,tmp);tmp.replace(p)\n\ndef sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\n\ndef initialize():\n    OUT.mkdir(parents=True,exist_ok=True);LOCAL.mkdir(parents=True,exist_ok=True)\n    source=json.loads((BASE/\'training_complete.json\').read_text())\n    assert source[\'parameters\']==227008\n    assert sha(BASE/\'adapter_best.pt\')==source[\'checkpoint_sha256\']\n    assert sha(BASE/\'normalization.npz\')==source[\'normalization_sha256\']\n    old=json.loads((BASE/\'config.json\').read_text())\n    assert old[\'source_sha256\']==sha(Path(__file__).parent/\'e9k_base_eval.py\'), \'Evaluation source differs from trained E9-k4\'\n    assert old[\'common_sha256\']==sha(Path(__file__).parent/\'common.py\')\n    assert old[\'tile_features_sha256\']==sha(CACHE/\'e8_tiles_cityscapes.npz\')\n    cfg=dict(CONFIG,source_sha256=sha(__file__),base_eval_sha256=sha(Path(__file__).parent/\'e9k_base_eval.py\'),\n        initial_checkpoint_sha256=source[\'checkpoint_sha256\'],normalization_sha256=source[\'normalization_sha256\'],\n        split_sha256=sha(BASE/\'splits.json\'),tile_features_sha256=old[\'tile_features_sha256\'])\n    p=OUT/\'config.json\'\n    if p.exists():assert json.loads(p.read_text())==cfg,\'Changed settings/source: use new E12_OUT/E12_LOCAL\'\n    else:write_json(p,cfg)\n    for name in [\'normalization.npz\',\'splits.json\',\'frame_controls.npz\',\'frame_controls.json\',\'frame_controls.csv\']:\n        dest=OUT/name\n        if not dest.exists():shutil.copy2(BASE/name,dest)\n    assert sha(OUT/\'normalization.npz\')==sha(BASE/\'normalization.npz\')\n    assert sha(OUT/\'splits.json\')==sha(BASE/\'splits.json\')\n    return cfg\n\n\ndef rank(a):return a.ravel().argsort().argsort().reshape(a.shape)\n\ndef paired(a,b,rng,repeats=1000):\n    diffs=[];n=len(a)\n    for _ in range(repeats):\n        idx=rng.integers(0,n,n);diffs.append(miou(a[idx].sum(0))-miou(b[idx].sum(0)))\n    ci=np.percentile(diffs,[2.5,97.5])\n    return {\'delta_miou_pp\':float(miou(a.sum(0))-miou(b.sum(0))),\'ci_low\':float(ci[0]),\'ci_high\':float(ci[1])}\n\n\ndef gate():\n    # All values read from actual reports; the E11 cost remains explicitly hypothetical.\n    data=SegSet(\'cs_test\');names=[p.stem for p,_ in data.items];n=len(names)\n    assert n==233 and names==[p.stem for p in sorted((CACHE/\'cs_test\').glob(\'*.npz\'))]\n    kroot=RESULTS.parent/\'e8c_k_sweep_v1\'\n    measured=json.loads((kroot/\'cost_measured.json\').read_text())\n    scenarios=json.loads((kroot/\'cost_scenarios.json\').read_text())[\'rows\']\n    c0=measured[\'models\'][\'b0\'][\'fvcore\'][\'counted_total_gops\'];c5=measured[\'models\'][\'b5\'][\'fvcore\'][\'counted_total_gops\']\n    with np.load(kroot/\'cs_test_progress.npz\') as z:\n        assert list(z[\'names\'])==names and int(z[\'completed\'])==n\n        teacher=z[\'confs\'];teacher_modes=list(z[\'modes\']);budgets=list(z[\'budgets\'])\n    with np.load(BASE/\'cs_test_progress.npz\') as z:\n        assert list(z[\'names\'])==names and int(z[\'completed\'])==n\n        initial=z[\'confs\'];initial_modes=list(z[\'modes\'])\n    frame=np.zeros((n,2,19,19),np.int64)\n    for i,name in enumerate(names):\n        with np.load(CACHE/\'cs_test\'/(name+\'.npz\')) as z:\n            frame[i,0]=confusion(z[\'pred_b0\'],z[\'gt\']);frame[i,1]=confusion(z[\'pred_b5\'],z[\'gt\'])\n    with np.load(CACHE/\'e8_tiles_cityscapes.npz\') as z:\n        h=z[\'cs_test__entropy__mean\'];k=z[\'cs_test__knn__mean\']\n    scores={\'raw_entropy\':h.mean((1,2)),\'rank_mean\':(rank(h)+rank(k)).mean((1,2))}\n    rows=[];matrices=[];labels=[];decisions=[];rng=np.random.default_rng(0)\n    for b in [.1,.2]:\n        scenario=next(r for r in scenarios if r[\'k\']==4 and r[\'budget\']==b)\n        current=scenario[\'hybrid_planning_count_halo_aware\']\n        # User-proposed E11 arithmetic: regionalize all other B5 parts with the same halo fraction.\n        hypothetical=(c0+scenario[\'adapter_core_gmac_scenario\']+scenario[\'stage3_halo_aware_gmac\']+\n                      scenario[\'b5_non_stage3_counted_gops\']*scenario[\'halo_query_fraction\'])\n        j=budgets.index(b);true_k4=teacher[:,j,teacher_modes.index(\'k4\')]\n        start=initial[:,j,initial_modes.index(\'pred_context_pred_output\')]\n        floor_rows=[]\n        for scope,target in [(\'current_hybrid\',current),(\'hypothetical_e11\',hypothetical)]:\n            fraction=(target-c0)/c5;assert 0<=fraction<=1\n            for method,score in scores.items():\n                order=np.argsort(-score,kind=\'stable\')\n                for bracket,count in [(\'floor\',int(np.floor(fraction*n))),(\'ceil\',int(np.ceil(fraction*n)))]:\n                    mask=np.zeros(n,bool);mask[order[:count]]=True\n                    conf=np.where(mask[:,None,None],frame[:,1],frame[:,0])\n                    label=f\'{scope}_{method}_b{b}_{bracket}\'\n                    row={\'label\':label,\'budget\':b,\'scope\':scope,\'method\':method,\'bracket\':bracket,\n                         \'target_arithmetic_count\':target,\'actual_frame_count\':count,\'actual_frame_fraction\':count/n,\n                         \'actual_counted_gops\':c0+count/n*c5,\'frame_miou\':float(miou(conf.sum(0))),\n                         \'teacher_k4_miou\':float(miou(true_k4.sum(0))),\'initial_adapter_miou\':float(miou(start.sum(0)))}\n                    if bracket==\'floor\':\n                        row.update({\'teacher_vs_frame_\'+key:value for key,value in paired(true_k4,conf,rng).items()})\n                        if scope==\'hypothetical_e11\':floor_rows.append(row)\n                    rows.append(row);matrices.append(conf);labels.append(label)\n        # Heuristic research gate, not proof of feasibility: true-k4 must beat BOTH predeclared floor controls.\n        passed=all(r[\'teacher_vs_frame_ci_low\']>0 for r in floor_rows)\n        decisions.append({\'budget\':b,\'positive_hypothetical_reserve\':passed,\n                          \'checks\':[{\'method\':r[\'method\'],\'delta\':r[\'teacher_vs_frame_delta_miou_pp\'],\n                                     \'ci_low\':r[\'teacher_vs_frame_ci_low\'],\'ci_high\':r[\'teacher_vs_frame_ci_high\']} for r in floor_rows]})\n    np.savez_compressed(OUT/\'gate_frame_controls.npz\',confs=np.stack(matrices,axis=1),names=np.array(names),labels=np.array(labels))\n    report={\'allow_training\':any(d[\'positive_hypothetical_reserve\'] for d in decisions),\'decisions\':decisions,\'rows\':rows,\n            \'warning\':\'Current-hybrid costs are arithmetic approximations; hypothetical E11 additionally scales non-stage3 parts by halo without an implementation or measured quality. Gate uses reused cs_test and nominal exploratory intervals; it is not proof of equal-time advantage.\'}\n    write_json(OUT/\'gate.json\',report);base.csv_rows(OUT/\'gate.csv\',rows)\n    print(json.dumps({\'allow_training\':report[\'allow_training\'],\'decisions\':decisions,\'warning\':report[\'warning\']},indent=2),flush=True)\n\n\ndef require_gate():\n    r=json.loads((OUT/\'gate.json\').read_text())\n    assert r[\'allow_training\'],\'Гейт не дал уверенного запаса над покадровыми контролями. Обучение не запускается; разберите gate.json.\'\n\n\ndef teacher_path(name):return LOCAL/\'teacher_logits\'/(name+\'.npz\')\n\ndef map_path(name):\n    existing=LOCAL.parent/BASE.name/\'targets/cs_fit\'/(name+\'.npz\')\n    return existing if existing.is_file() else LOCAL/\'state_targets\'/(name+\'.npz\')\n\n\ndef load_local_targets(names):\n    for i,name in enumerate(names):\n        p=map_path(name)\n        if not p.exists():\n            src=BASE/\'targets/cs_fit\'/(name+\'.npz\')\n            assert src.is_file(),f\'Нет предыдущих teacher-целей {src}; восстанови E9-k4 targets\'\n            p.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(src,p)\n        if not teacher_path(name).exists():\n            p=teacher_path(name);p.parent.mkdir(parents=True,exist_ok=True)\n            shutil.copy2(OUT/\'teacher_logits\'/p.name,p)\n        if (i+1)%50==0:print(\'Восстановлены цели:\',i+1,\'/\',len(names),flush=True)\n\n@torch.inference_mode()\ndef prepare():\n    require_gate()\n    splits=json.loads((OUT/\'splits.json\').read_text());names=splits[\'train\']+splits[\'val\']\n    data=SegSet(\'cs_fit\');index={p.stem:i for i,(p,_) in enumerate(data.items)}\n    assert set(names)==set(index) and len(names)==267\n    model=load_model(\'b5\').requires_grad_(False);start=time.monotonic()\n    for i,name in enumerate(names):\n        p=teacher_path(name);backup=OUT/\'teacher_logits\'/p.name\n        p.parent.mkdir(parents=True,exist_ok=True);backup.parent.mkdir(parents=True,exist_ok=True)\n        if not p.exists() and backup.exists():shutil.copy2(backup,p)\n        if not p.exists():\n            img=data[index[name]][0][None].cuda().half()\n            logits=model(pixel_values=img).logits\n            pred=F.interpolate(logits.float(),SIZE,mode=\'bilinear\',align_corners=False).argmax(1)[0].byte().cpu().numpy()\n            with np.load(CACHE/\'cs_fit\'/(name+\'.npz\')) as z:assert (pred==z[\'pred_b5\']).mean()>.999\n            tmp=p.with_suffix(\'.npz.part\')\n            with tmp.open(\'wb\') as f:np.savez_compressed(f,logits=logits[0].cpu().numpy())\n            tmp.replace(p)\n        with np.load(p) as z:assert z[\'logits\'].shape==(19,128,256) and np.isfinite(z[\'logits\']).all()\n        if not backup.exists():\n            tmp=backup.with_suffix(\'.npz.part\');shutil.copy2(p,tmp);tmp.replace(backup)\n        if (i+1)%10==0 or i+1==len(names):print(f\'Teacher logits: {i+1}/{len(names)}, {(time.monotonic()-start)/60:.1f} мин\',flush=True)\n    load_local_targets(names)\n    write_json(OUT/\'prepare_complete.json\',{\'complete\':True,\'frames\':len(names),\'source_states\':str(BASE/\'targets/cs_fit\')})\n\n\nclass InjectBlock(nn.Module):\n    def __init__(self,layer,state,i,n,use_checkpoint=True):\n        super().__init__();self.layer=layer;self.state=state;self.i=i;self.n=n;self.use_checkpoint=use_checkpoint\n    def forward(self,x,*args,**kwargs):\n        s=self.state\n        if s.mask is None:return self.layer(x,*args,**kwargs)\n        # Capture immutable per-call tensor references BEFORE state resets. Backward replay never reads state.\n        context=s.contexts[CONTEXTS[self.i*4//self.n]];output=s.output;mask=s.mask\n        layer=self.layer;last=self.i==self.n-1\n        def core(y,c,o,m):\n            out=layer(torch.where(m,y,c),*args,**kwargs)\n            value=torch.where(m,out[0],c)\n            if last:value=torch.where(m,value,o)\n            return (value,)+tuple(out[1:])\n        if self.use_checkpoint and torch.is_grad_enabled():\n            return checkpoint(core,x,context,output,mask,use_reentrant=False,preserve_rng_state=False)\n        return core(x,context,output,mask)\n\nclass CheckpointBlock(nn.Module):\n    def __init__(self,layer):super().__init__();self.layer=layer\n    def forward(self,x,*args,**kwargs):\n        layer=self.layer\n        def core(y):return layer(y,*args,**kwargs)\n        if torch.is_grad_enabled() and x.requires_grad:\n            return checkpoint(core,x,use_reentrant=False,preserve_rng_state=False)\n        return core(x)\n\nclass CheckpointHead(nn.Module):\n    def __init__(self,head):super().__init__();self.head=head\n    def forward(self,features):\n        head=self.head\n        def core(*xs):return head(xs)\n        if torch.is_grad_enabled() and any(x.requires_grad for x in features):\n            return checkpoint(core,*features,use_reentrant=False,preserve_rng_state=False)\n        return core(*features)\n\n\ndef install_student(model,use_checkpoint=True):\n    state=SimpleNamespace(mask=None,contexts=None,output=None)\n    blocks=model.segformer.encoder.block[2]\n    assert len(blocks)==40\n    for i,block in enumerate(blocks):blocks[i]=InjectBlock(block,state,i,len(blocks),use_checkpoint)\n    if use_checkpoint:\n        blocks4=model.segformer.encoder.block[3]\n        for i,block in enumerate(blocks4):blocks4[i]=CheckpointBlock(block)\n        model.decode_head=CheckpointHead(model.decode_head)\n    return state\n\n\ndef student_logits(model,state,image,mask,states):\n    # Deliberately NO inference_mode/no_grad. Frozen parameters still transmit derivatives to adapter inputs.\n    state.mask=mask;state.contexts=states;state.output=states[\'final\']\n    try:return model(pixel_values=image).logits\n    finally:state.mask=None;state.contexts=None;state.output=None\n\n\ndef decode_states(output,norm,hidden_channels=320):\n    states={}\n    for j,k in enumerate(STATES):\n        a=output[:,j*hidden_channels:(j+1)*hidden_channels]\n        a=a*norm[k+\'_std\'][None,:,None,None]+norm[k+\'_mean\'][None,:,None,None]\n        states[k]=a.flatten(2).transpose(1,2).to(dtype=norm[\'b5_dtype\'])\n    return states\n\n\ndef pixel_kl(student,teacher,pixels,size=SIZE,temperature=2.):\n    # Teacher detached; score exactly the selected working-resolution pixels, including region boundaries.\n    s=F.interpolate(student.float(),size,mode=\'bilinear\',align_corners=False).permute(0,2,3,1)[pixels]\n    t=F.interpolate(teacher.detach().float(),size,mode=\'bilinear\',align_corners=False).permute(0,2,3,1)[pixels]\n    assert len(s)>0\n    return F.kl_div(F.log_softmax(s/temperature,dim=-1),F.softmax(t/temperature,dim=-1),reduction=\'batchmean\')*temperature**2\n\n\ndef fixed_rng(name,epoch,tag):\n    value=int(hashlib.sha256(f\'{CONFIG["seed"]}:{name}:{epoch}:{tag}\'.encode()).hexdigest()[:16],16)\n    return np.random.default_rng(value)\n\n\ndef random_mask(name,epoch,fraction=None):\n    rng=fixed_rng(name,epoch,\'val\'+str(fraction) if fraction is not None else \'train\')\n    b=fraction if fraction is not None else rng.uniform(*CONFIG[\'train_mask_fraction\'])\n    mask=np.zeros(128,bool);mask[rng.permutation(128)[:max(1,round(b*128))]]=True\n    return mask.reshape(8,16)\n\n\ndef load_sample(name,norm,device,image_index,data):\n    image=data[image_index[name]][0][None].to(device=device,dtype=norm[\'b5_dtype\'])\n    with np.load(map_path(name)) as z:\n        x=torch.from_numpy(z[\'x\'].astype(np.float32))[None].to(device)\n        x=(x-norm[\'x_mean\'][None,:,None,None])/norm[\'x_std\'][None,:,None,None]\n        target=torch.from_numpy(np.concatenate([(z[k].astype(np.float32)-norm[k+\'_mean_np\'][:,None,None])/norm[k+\'_std_np\'][:,None,None] for k in STATES]))[None].to(device)\n    with np.load(teacher_path(name)) as z:teacher=torch.from_numpy(z[\'logits\'].astype(np.float32))[None].to(device)\n    return image,x,target,teacher\n\n\ndef loss_for(adapter,model,state,sample,norm,tile_mask):\n    image,x,target,teacher=sample\n    native=torch.as_tensor(tile_mask,device=x.device).repeat_interleave(4,0).repeat_interleave(4,1).reshape(1,2048,1)\n    pixels=torch.as_tensor(tile_mask,device=x.device).repeat_interleave(64,0).repeat_interleave(64,1)[None]\n    output=adapter(x.float());states=decode_states(output,norm)\n    student=student_logits(model,state,image,native,states)\n    kl=pixel_kl(student,teacher,pixels,temperature=CONFIG[\'temperature\'])\n    mse=base.normalized_loss(output,target)\n    return kl+CONFIG[\'state_weight\']*mse,kl,mse\n\n\ndef make_norm(device):\n    with np.load(OUT/\'normalization.npz\') as z:arrays={k:z[k] for k in z.files}\n    result={k:torch.as_tensor(v,device=device) for k,v in arrays.items()}\n    for k in STATES:result[k+\'_mean_np\']=arrays[k+\'_mean\'];result[k+\'_std_np\']=arrays[k+\'_std\']\n    result[\'b5_dtype\']=torch.float16 if str(device).startswith(\'cuda\') else torch.float32\n    return result\n\n@torch.no_grad()\ndef validate(adapter,model,state,names,norm,device,index,data):\n    adapter.eval();sums=np.zeros(3);count=0\n    for name in names:\n        sample=load_sample(name,norm,device,index,data)\n        for b in CONFIG[\'val_mask_fractions\']:\n            values=loss_for(adapter,model,state,sample,norm,random_mask(name,0,b))\n            sums+=np.array([float(v.item()) for v in values]);count+=1\n    return dict(zip([\'loss\',\'kl\',\'state_mse\'],(sums/count).tolist()))\n\n\ndef base_unchanged(model,versions,buffers):\n    assert all(not p.requires_grad and p.grad is None and p._version==versions[n] for n,p in model.named_parameters()),\'B5 parameter changed or received gradients\'\n    assert all(torch.equal(v,buffers[n]) for n,v in model.named_buffers()),\'B5 buffer changed\'\n\n\ndef train(device="cuda"):\n    require_gate()\n    if (OUT/\'training_complete.json\').exists():\n        r=json.loads((OUT/\'training_complete.json\').read_text())\n        assert r[\'checkpoint_sha256\']==sha(OUT/\'adapter_best.pt\')\n        print(\'Дообучение уже завершено, пропускаем\',flush=True);return\n    splits=json.loads((OUT/\'splits.json\').read_text());load_local_targets(splits[\'train\']+splits[\'val\'])\n    torch.manual_seed(CONFIG[\'seed\']);np.random.seed(CONFIG[\'seed\'])\n    adapter=base.Adapter().to(device)\n    adapter.load_state_dict(torch.load(BASE/\'adapter_best.pt\',map_location=device,weights_only=True)[\'model\'])\n    model=load_model(\'b5\',device=device).requires_grad_(False).eval();state=install_student(model);model.eval()\n    versions={n:p._version for n,p in model.named_parameters()};buffers={n:b.detach().clone() for n,b in model.named_buffers()}\n    assert not set(map(id,adapter.parameters()))&set(map(id,model.parameters()))\n    optimizer=torch.optim.AdamW(adapter.parameters(),lr=CONFIG[\'lr\'],weight_decay=CONFIG[\'weight_decay\'])\n    scaler=torch.amp.GradScaler(\'cuda\',init_scale=CONFIG[\'amp_initial_scale\'],enabled=str(device).startswith(\'cuda\'))\n    norm=make_norm(device);data=SegSet(\'cs_fit\');index={p.stem:i for i,(p,_) in enumerate(data.items)}\n    last=OUT/\'adapter_last.pt\';start_epoch=0;bad=0;history=[];best_epoch=0;best=float(\'inf\');best_weights=None\n    if last.exists():\n        r=torch.load(last,map_location=device,weights_only=True)\n        adapter.load_state_dict(r[\'model\']);optimizer.load_state_dict(r[\'optimizer\']);scaler.load_state_dict(r[\'scaler\'])\n        start_epoch=r[\'epoch\'];bad=r[\'bad\'];history=r[\'history\'];best_epoch=r[\'best_epoch\'];best=r[\'best\']\n        best_weights={k:v.detach().cpu().clone() for k,v in r[\'best_model\'].items()}\n        atomic_torch(OUT/\'adapter_best.pt\',{\'model\':best_weights,\'epoch\':best_epoch,\'val_loss\':best})\n        print(\'Продолжаем после эпохи\',start_epoch,flush=True)\n    if not last.exists():\n        baseline=validate(adapter,model,state,splits[\'val\'],norm,device,index,data)\n        best=baseline[\'loss\'];history=[{\'epoch\':0,\'val\':baseline,\'source\':\'initial E9-k4; no update\'}]\n        best_weights={k:v.detach().cpu().clone() for k,v in adapter.state_dict().items()}\n        atomic_torch(OUT/\'adapter_best.pt\',{\'model\':best_weights,\'epoch\':0,\'val_loss\':best})\n        write_json(OUT/\'initial_validation.json\',baseline);write_json(OUT/\'training_history.json\',history)\n        print(\'Исходный адаптер, эпоха 0:\',baseline,flush=True)\n    # One backward without optimizer.step: confirm gradient path and memory before starting epochs.\n    if str(device).startswith(\'cuda\'):torch.cuda.reset_peak_memory_stats()\n    optimizer.zero_grad(set_to_none=True);adapter.train()\n    sample=load_sample(splits[\'train\'][0],norm,device,index,data)\n    try:\n        loss,_,_=loss_for(adapter,model,state,sample,norm,random_mask(splits[\'train\'][0],0,.1))\n        scaler.scale(loss).backward();scaler.unscale_(optimizer)\n        assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in adapter.parameters()), \'Nonfinite adapter gradients in probe\'\n        grads=adapter.net[2].weight.grad\n        norms=[float(grads[j*320:(j+1)*320].float().norm().item()) for j in range(5)]\n        assert all(np.isfinite(v) and v>0 for v in norms),\'Missing/nonfinite adapter head gradients\'\n        base_unchanged(model,versions,buffers)\n        write_json(OUT/\'backward_probe.json\',{\'passed\':True,\'head_gradient_norms\':norms,\n            \'peak_allocated_mib\':torch.cuda.max_memory_allocated()/2**20 if str(device).startswith(\'cuda\') else None,\'b5_unchanged\':True})\n        print(\'Backward probe OK:\',norms,flush=True)\n        scaler.update();optimizer.zero_grad(set_to_none=True)\n        del loss,sample\n    except torch.cuda.OutOfMemoryError:\n        write_json(OUT/\'backward_probe.json\',{\'passed\':False,\'reason\':\'CUDA OOM; no optimizer update performed\'})\n        raise\n    for epoch in range(start_epoch,CONFIG[\'epochs\']):\n        if bad>=CONFIG[\'patience\']:break\n        adapter.train();order=np.random.default_rng(CONFIG[\'seed\']+epoch).permutation(len(splits[\'train\']))\n        sums=np.zeros(3);count=0;skipped=0;start=time.monotonic()\n        for step,i in enumerate(order):\n            name=splits[\'train\'][int(i)];sample=load_sample(name,norm,device,index,data)\n            optimizer.zero_grad(set_to_none=True)\n            loss,kl,mse=loss_for(adapter,model,state,sample,norm,random_mask(name,epoch+1))\n            assert torch.isfinite(loss),\'Nonfinite loss\'\n            scaler.scale(loss).backward();scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(adapter.parameters(),CONFIG[\'clip_norm\'])\n            scale_before=scaler.get_scale();scaler.step(optimizer);scaler.update()\n            skipped+=int(scaler.get_scale()<scale_before)\n            sums+=np.array([loss.item(),kl.item(),mse.item()]);count+=1\n            if (step+1)%25==0:print(f\'Эпоха {epoch+1}: {step+1}/{len(order)}, {(time.monotonic()-start)/60:.1f} мин\',flush=True)\n        validation=validate(adapter,model,state,splits[\'val\'],norm,device,index,data)\n        assert np.isfinite(validation[\'loss\']), \'Nonfinite validation loss\'\n        row={\'epoch\':epoch+1,\'train\':dict(zip([\'loss\',\'kl\',\'state_mse\'],(sums/count).tolist())),\'val\':validation,\n             \'amp_skipped_steps\':skipped,\'amp_scale\':scaler.get_scale()}\n        history.append(row)\n        if validation[\'loss\']<best:\n            best=validation[\'loss\'];best_epoch=epoch+1;bad=0\n            best_weights={k:v.detach().cpu().clone() for k,v in adapter.state_dict().items()}\n            atomic_torch(OUT/\'adapter_best.pt\',{\'model\':best_weights,\'epoch\':best_epoch,\'val_loss\':best})\n        else:bad+=1\n        base_unchanged(model,versions,buffers)\n        atomic_torch(last,{\'model\':adapter.state_dict(),\'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict(),\n                          \'epoch\':epoch+1,\'best\':best,\'best_epoch\':best_epoch,\'bad\':bad,\'history\':history,\'best_model\':best_weights})\n        write_json(OUT/\'training_history.json\',history);print(row,flush=True)\n    base_unchanged(model,versions,buffers)\n    write_json(OUT/\'training_complete.json\',{\'complete\':True,\'best_epoch\':best_epoch,\'val_loss\':best,\n        \'parameters\':227008,\'checkpoint_sha256\':sha(OUT/\'adapter_best.pt\'),\n        \'normalization_sha256\':sha(OUT/\'normalization.npz\'),\'initial_checkpoint_sha256\':sha(BASE/\'adapter_best.pt\'),\n        \'b5_parameters_and_buffers_unchanged\':True,\'train_frames\':len(splits[\'train\']),\'val_frames\':len(splits[\'val\'])})\n\n\ndef evaluate():\n    require_gate();base.evaluate()\n    rows=json.loads((OUT/\'routing.json\').read_text())[\'cs_test\']\n    old=json.loads((BASE/\'routing.json\').read_text())[\'cs_test\']\n    for row in rows:\n        initial=next(r[\'miou\'] for r in old if r[\'budget\']==row[\'budget\'] and r[\'variant\']==\'pred_context_pred_output\')\n        row[\'initial_k4_pred_all_miou\']=initial;row[\'delta_vs_initial_k4_pp\']=row[\'miou\']-initial\n    write_json(OUT/\'routing.json\',dict(json.loads((OUT/\'routing.json\').read_text()),cs_test=rows))\n    base.csv_rows(OUT/\'routing.csv\',rows)\n\n\ndef bootstrap():\n    with np.load(OUT/\'cs_test_progress.npz\') as z:\n        ours=z[\'confs\'];names=z[\'names\'];budgets=list(z[\'budgets\']);modes=list(z[\'modes\']);assert int(z[\'completed\'])==len(names)\n    with np.load(BASE/\'cs_test_progress.npz\') as z:\n        initial=z[\'confs\'];assert np.array_equal(z[\'names\'],names);initial_modes=list(z[\'modes\'])\n    with np.load(OUT/\'gate_frame_controls.npz\') as z:\n        gate_confs=z[\'confs\'];labels=list(z[\'labels\']);assert np.array_equal(z[\'names\'],names)\n    meta=json.loads((OUT/\'gate.json\').read_text())[\'rows\'];rng=np.random.default_rng(0);rows=[]\n    for unit in [\'frame\',\'sequence\']:\n        groups=np.arange(len(names)).astype(str) if unit==\'frame\' else np.array([\'_\'.join(n.split(\'_\')[:2]) for n in names])\n        unique=sorted(set(groups))\n        def group(c):return np.stack([c[groups==g].sum(0) for g in unique])\n        for j,b in enumerate(budgets):\n            a=group(ours[:,j,modes.index(\'pred_context_pred_output\')])\n            references={\'initial_k4\':initial[:,j,initial_modes.index(\'pred_context_pred_output\')],\n                        \'true_k4\':ours[:,j,modes.index(\'true_context_true_output\')],\n                        \'full_tiles\':ours[:,j,modes.index(\'full_tiles\')]}\n            for item in meta:\n                if item[\'budget\']==b and item[\'bracket\']==\'floor\':references[item[\'label\']]=gate_confs[:,labels.index(item[\'label\'])]\n            for label,c in references.items():\n                result=paired(a,group(c),rng)\n                rows.append({\'budget\':b,\'unit\':unit,\'groups\':len(unique),\'comparison\':\'finetuned_pred_all minus \'+label,**result,\n                    \'scope\':\'quality contrast; cost-matched labels use speculative arithmetic, not measured time\'})\n    write_json(OUT/\'bootstrap.json\',rows);base.csv_rows(OUT/\'bootstrap.csv\',rows)\n    for row in rows:print(row,flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'stage\',choices=[\'gate\',\'prepare\',\'train\',\'evaluate\',\'bootstrap\']);args=p.parse_args()\n    initialize()\n    if args.stage not in [\'gate\',\'bootstrap\']:assert torch.cuda.is_available(),\'Select GPU runtime\'\n    import platform,transformers\n    write_json(OUT/\'environment.json\',{\'python\':platform.python_version(),\'torch\':torch.__version__,\n        \'transformers\':transformers.__version__,\'gpu\':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})\n    globals()[args.stage]()\n'}
for name,text in NEW_SOURCES.items():
    for path in [EXP/name,OUT/name]:
        if path.exists():
            assert path.read_text()==text, 'Изменилась версия кода: используй новый OUT/LOCAL.'
        else:path.write_text(text)


In [ ]:
# 8. Запускатель с логом на Drive.
def run_e12(stage):
    log_path=OUT/f'{stage}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('Запуск:',stage,'| Лог:',log_path,flush=True)
    start=time.monotonic()
    with log_path.open('w') as log:
        process=subprocess.Popen([sys.executable,'-u',str(EXP/'e12_output_distill.py'),stage],
                                 cwd=EXP,env=os.environ.copy(),stdout=log,stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    status=process.wait(timeout=30);break
                except subprocess.TimeoutExpired:
                    tail=log_path.read_text(errors='replace').splitlines()[-2:]
                    print(f'{stage}: {(time.monotonic()-start)/60:.1f} мин | '+' | '.join(tail),flush=True)
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    print(log_path.read_text(errors='replace')[-16000:])
    assert status==0, f'Ошибка {status}. Смотри {log_path}'
    print('Завершено за',round((time.monotonic()-start)/60,1),'мин')


## CPU-гейт перед GPU-обучением

Печатаются два явно разных сценария:
- `current_hybrid`: прежняя расчётная цена с полнокадровыми частями B5 вне stage 3.
- `hypothetical_e11`: предложение регионализировать все эти части, умножив их цену на halo-долю. Это предположение, а не реализованная архитектура; качество такой замены не измерено.

Для обоих считаются raw_entropy/rank_mean на соседних целых количествах кадров. Основной контроль — floor, укладывающийся в арифметический бюджет; ceil показывается отдельно. Значения берутся из реальных отчётов и кэша, не из захардкоженных приростов.

Обучение разрешается, если хотя бы для одного гипотетического E11-бюджета нижняя номинальная граница 95% парного CI для `настоящий k4 − кадровый контроль floor` выше нуля для обоих заранее выбранных контролей. Это консервативный исследовательский гейт, не доказательство реальной вычислительной выгоды и не строгий предел возможного обучаемого контекста.

Если `allow_training=False`, следующие шаги автоматически останавливаются. Не меняй критерий после просмотра результата, чтобы всё-таки начать обучение. При положительном только гипотетическом гейте дообучение исследует потенциал будущей архитектуры; текущая полнокадровая схема от этого не становится выгодной.


In [ ]:
# 9. Только готовый кэш; новые GPU-прогоны не нужны.
run_e12('gate')
import pandas as pd
from IPython.display import display
gate=json.loads((OUT/'gate.json').read_text())
display(pd.read_csv(OUT/'gate.csv'))
print('Разрешено ограниченное обучение:',gate['allow_training'])
print(gate['warning'])


In [ ]:
# 10. Только если гейт пройден: cache логитов обычной B5 на Frankfurt.
assert json.loads((OUT/'gate.json').read_text())['allow_training'], 'Гейт не пройден. Остановись и пришли gate.json.'
run_e12('prepare')


## Градиент и память

Stage 3, stage 4 и decoder head используют non-reentrant gradient checkpointing. Контекст, final и маска передаются в checkpoint явными тензорами; повторное вычисление в backward не читает изменяемое состояние обёртки после его сброса.

Перед эпохами выполняется один backward **без optimizer.step**. Проверяется ненулевой конечный градиент всех пяти выходных групп и отсутствие градиентов/изменений параметров и буферов B5. Результат и пик памяти — в `backward_probe.json`. Никакой LoRA или дообучения весов B5 нет.

Используется fp16 B5 и fp32 адаптер с GradScaler (начальный scale 1024). В истории сохраняются AMP-пропуски шагов и scale. Если probe выдаст OOM или неконечные градиенты, не меняй разрешение и маски на ходу; пришли лог. Реальный GPU-probe ещё должен выполниться в твоём окружении.


In [ ]:
# 11. Дообучение максимум 5 эпох; лучшим может остаться исходный epoch=0.
run_e12('train')
print((OUT/'training_complete.json').read_text())
display(pd.json_normalize(json.loads((OUT/'training_history.json').read_text())))


In [ ]:
# 12. Те же 5 диагностических вариантов на cs_test: teacher/teacher, pred/teacher,
# teacher/pred, pred/pred, постоянные средние. Веса после train уже зафиксированы.
run_e12('evaluate')
display(pd.read_csv(OUT/'routing.csv'))


In [ ]:
# 13. Парный bootstrap: исходный E9-k4, настоящий k4, full_tiles и кадровые floor-контроли.
run_e12('bootstrap')
display(pd.read_csv(OUT/'bootstrap.csv'))


In [ ]:
# 14. Итоговый архив: без тяжёлых teacher-логитов/целей и без last checkpoint.
import zipfile
archive=OUT/'e12_output_distill_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file() and p.name not in {archive.name,'adapter_last.pt'} and not p.name.endswith('.part'):
            z.write(p,p.name)
    for p in [BASE/'config.json',BASE/'training_complete.json',BASE/'routing.json',
              SAVE/'e8c_k_sweep_v1/e8c_quality.json',SAVE/'e8c_k_sweep_v1/cost_scenarios.json',EXP/'common.py']:
        if p.is_file():z.write(p,'reference/'+p.name)
print('Пришли:',archive)
# Скачать:
# from google.colab import files
# files.download(str(archive))


## Что должно ответить исследование

Основной контраст — finetuned `pred_context_pred_output` против исходного E9-k4 на тех же кадрах/масках. Затем смотрим потерю к настоящему k4 и к full_tiles. Более низкая validation-потеря ещё не гарантирует рост mIoU.

Покадровые сравнения разделены на текущую и гипотетическую стоимость. Они не доказывают преимущество при одинаковом полном времени. В исходном E9-k4 разница с настоящими состояниями составляла примерно −1.42/−1.15 п.п.; проверяем, сокращает ли именно output-loss эту потерю.

Не выбираем эпоху или новый коэффициент по cs_test. При отрицательном результате фиксируем текущую попытку; новые варианты будут отдельными гипотезами.

При сбросе VM выполняй notebook с начала: E8-кэш восстановится, targets возьмутся из прежнего E9-k4, новые teacher-логиты — с Drive. Обучение продолжается с последней завершённой эпохи, включая optimizer/scaler и согласованный лучший checkpoint. Незавершённая эпоха считается заново. Оценка cs_test продолжается с checkpoint каждые 10 кадров.
